# Kaggle AI Dubbing Studio

Upload video or audio, transcribe and diarize it, use Step 1 to correct the source transcript with AI, then Step 2 to translate into a VoxCPM2-supported language before generating the dub.

**One page, no database:** all review and controls are in one Gradio screen. Session files are temporary under `/kaggle/working`.

## Before running

1. In Kaggle, enable **GPU** and **Internet** in Notebook options.
2. Accept access to `pyannote/speaker-diarization-community-1` on Hugging Face.
3. Add Kaggle Secrets named `HF_TOKEN` and `GROQ_API_KEY`.
4. Run the cells from top to bottom. VoxCPM2 loads when measuring pace, translating with automatic measurement, or generating a dub.

The pipeline uses Demucs for vocal/background separation, faster-whisper for transcription, pyannote for anonymous speaker IDs, Groq for multilingual translation, and VoxCPM2 for speech generation in its 30 supported languages. Per-speaker reference clips are extracted from the separated vocals when enough speech is available. You can optionally provide one shared reference clip to override them.

> Voice cloning quality depends on clear reference audio. The generated voices are not verified real-person identities. Please use media and reference audio you have permission to process.

In [ ]:
%pip install -q "av<19" "faster-whisper>=1.1,<2" "pyannote.audio==4.0.7" "demucs>=4,<5" "groq>=0.18,<2" "gradio>=4,<6" "nvidia-cublas-cu12==12.9.2.10" soundfile scipy librosa
%pip install -q -U voxcpm

# Install optional FunASR without changing Torch or its dependencies.
import subprocess
import sys
_optional_funasr = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "funasr>=1.2,<2"],
    capture_output=True,
    text=True,
)
if _optional_funasr.returncode:
    print("Optional SenseVoice detection is unavailable; manual keep-original ranges still work.")
else:
    print("Optional SenseVoice installed without changing existing dependencies.")
print("Dependencies installed. Restart the Kaggle session, then run all cells from the top.")

In [ ]:
import ctypes
import gc
import json
import math
import os
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
import gradio as gr
from scipy.signal import resample_poly

from kaggle_secrets import UserSecretsClient


def kaggle_secret(name):
    try:
        return UserSecretsClient().get_secret(name)
    except Exception:
        return ""


HF_TOKEN = kaggle_secret("HF_TOKEN")
GROQ_API_KEY = kaggle_secret("GROQ_API_KEY")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
GROQ_MODEL = "openai/gpt-oss-120b"
DIARIZATION_MODEL = "pyannote/speaker-diarization-community-1"
MAX_LINE_CHARS = 500
OUTPUT_ROOT = Path("/kaggle/working/ai_dubbing_studio")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FFMPEG = shutil.which("ffmpeg")
FFPROBE = shutil.which("ffprobe")

if not FFMPEG:
    raise RuntimeError("FFmpeg was not found. Enable Internet and install FFmpeg in the Kaggle session.")

try:
    import nvidia.cublas

    cublas_dir = Path(nvidia.cublas.__path__[0]) / "lib"
    cublas_library = cublas_dir / "libcublas.so.12"
    if cublas_library.is_file():
        os.environ["LD_LIBRARY_PATH"] = str(cublas_dir) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
        ctypes.CDLL(str(cublas_library), mode=ctypes.RTLD_GLOBAL)
        print("CUDA cuBLAS loaded for faster-whisper.")
    else:
        print("CUDA cuBLAS library is missing; faster-whisper will use its CPU fallback.")
except (ImportError, OSError) as error:
    print(f"CUDA cuBLAS preload unavailable; faster-whisper may use CPU: {error}")

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available (models will use CPU)")
print("HF_TOKEN:", "available" if HF_TOKEN else "missing")
print("GROQ_API_KEY:", "available" if GROQ_API_KEY else "missing")
print("Secrets are read from Kaggle Secrets and are not written to project files.")

In [ ]:
VIDEO_EXTENSIONS = {".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v"}
SAMPLE_RATE = 44100
REFERENCE_LIMIT_SECONDS = 10.0
REFERENCE_TARGET_SECONDS = 5.0
MIN_SPEAKER_TURN_SECONDS = 0.25


def run_command(command, label):
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode:
        detail = (result.stderr or result.stdout or "No process output")[-2500:]
        raise RuntimeError(f"{label} failed:\n{detail}")
    return result.stdout


def _probe_video_duration(media_path):
    if not FFPROBE:
        raise RuntimeError("ffprobe is required to align the final audio to the video duration.")
    result = run_command(
        [FFPROBE, "-v", "error", "-select_streams", "v:0", "-show_entries",
         "stream=duration:format=duration", "-of", "json", str(media_path)],
        "Video duration probe",
    )
    try:
        metadata = json.loads(result)
        candidates = [metadata.get("streams", [{}])[0].get("duration"),
                      metadata.get("format", {}).get("duration")]
        duration = next(float(value) for value in candidates
                        if value not in (None, "N/A") and math.isfinite(float(value)) and float(value) > 0)
    except (json.JSONDecodeError, KeyError, IndexError, StopIteration, TypeError, ValueError) as error:
        raise RuntimeError(f"Could not determine video duration for sync: {result!r}") from error
    return duration


def extract_audio(media_path, job_dir):
    media_path = Path(media_path)
    if not media_path.is_file():
        raise FileNotFoundError(f"Uploaded media was not found: {media_path}")
    audio_path = job_dir / "original_audio.wav"
    run_command(
        [
            FFMPEG, "-y", "-v", "error", "-i", str(media_path),
            "-map", "0:a:0", "-vn", "-af", "aresample=async=1:first_pts=0,asetpts=PTS-STARTPTS",
            "-ac", "2", "-ar", str(SAMPLE_RATE),
            "-c:a", "pcm_s16le", str(audio_path),
        ],
        "Audio extraction",
    )
    return audio_path, media_path.suffix.lower() in VIDEO_EXTENSIONS


def separate_vocals(audio_path, job_dir, progress):
    output_dir = job_dir / "separated"
    command = [
        sys.executable, "-m", "demucs", "--two-stems=vocals",
        "-n", "htdemucs", "-o", str(output_dir), str(audio_path),
    ]
    progress(0.18, desc="Separating vocals and background with Demucs")
    run_command(command, "Demucs separation")
    vocal_matches = list(output_dir.rglob("vocals.wav"))
    background_matches = list(output_dir.rglob("no_vocals.wav"))
    if not vocal_matches or not background_matches:
        raise RuntimeError("Demucs finished but its vocals/background WAV files were not found.")
    return vocal_matches[0], background_matches[0]


def transcribe(vocals_path, language, model_size, progress):
    from faster_whisper import WhisperModel

    requested_language = None if language == "Auto-detect" else language

    def run_model(device, compute_type):
        model = WhisperModel(model_size, device=device, compute_type=compute_type)
        try:
            segments, info = model.transcribe(
                str(vocals_path),
                language=requested_language,
                vad_filter=True,
                beam_size=5,
                condition_on_previous_text=False,
            )
            rows = [
                {
                    "start": float(segment.start),
                    "end": float(segment.end),
                    "text": segment.text.strip(),
                }
                for segment in segments
                if segment.text.strip()
            ]
            return rows, info.language
        finally:
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    progress(0.35, desc="Transcribing dialogue with faster-whisper")
    if torch.cuda.is_available():
        try:
            return run_model("cuda", "float16")
        except Exception as error:
            print(f"Whisper GPU inference failed; retrying on CPU. Details: {error}")
    return run_model("cpu", "int8")


def diarize(vocals_path, progress):
    if not HF_TOKEN:
        raise ValueError("HF_TOKEN is missing. Add it under Kaggle Notebook options > Secrets.")
    from pyannote.audio import Pipeline

    progress(0.53, desc="Detecting speaker turns with pyannote")
    pipeline = Pipeline.from_pretrained(DIARIZATION_MODEL, token=HF_TOKEN)
    if pipeline is None:
        raise RuntimeError(f"Could not load {DIARIZATION_MODEL}; check Hugging Face access and HF_TOKEN.")
    use_cuda = torch.cuda.is_available()
    if use_cuda:
        pipeline.to(torch.device("cuda"))
    try:
        result = pipeline(str(vocals_path))
        turns = [
            {
                "start": float(turn.start),
                "end": float(turn.end),
                "speaker": str(speaker),
            }
            for turn, speaker in result.speaker_diarization
        ]
    finally:
        del pipeline
        gc.collect()
        if use_cuda:
            torch.cuda.empty_cache()
    if not turns:
        raise RuntimeError("pyannote did not detect any speaker turns.")
    return turns


def is_numeric_fragment(text):
    chars = [char for char in text if not char.isspace()]
    return bool(chars) and all(char.isdigit() or char in ".,-" for char in chars)


def split_sentences(text):
    endings = set(".?!…。！？।॥")
    closers = set("\"'”’)]}»")
    sentences = []
    start = 0
    index = 0
    while index < len(text):
        if text[index] not in endings:
            index += 1
            continue
        end = index + 1
        while end < len(text) and (text[end] in endings or text[end] in closers):
            end += 1
        if end == len(text) or text[end].isspace():
            candidate = text[start:end].strip()
            if candidate and not is_numeric_fragment(candidate):
                sentences.append(candidate)
                start = end
                while start < len(text) and text[start].isspace():
                    start += 1
                index = start
            else:
                index = end
        else:
            index += 1
    remainder = text[start:].strip()
    if remainder:
        sentences.append(remainder)
    return sentences


def split_long_text(text, limit=MAX_LINE_CHARS):
    text = text.strip()
    if len(text) <= limit:
        return [text] if text else []
    priorities = (
        set(".?!…。！？।॥"),
        set(";:；："),
        set(",，"),
        set("—–-"),
    )
    for punctuation in priorities:
        candidates = []
        for index, char in enumerate(text[:limit]):
            if char not in punctuation:
                continue
            end = index + 1
            while end < len(text) and text[end] in ".?!…。！？।॥\"'”’)]}»":
                end += 1
            if end <= limit and (end == len(text) or text[end].isspace()):
                candidates.append(end)
        if candidates:
            split_at = candidates[-1]
            break
    else:
        split_at = max((i for i, char in enumerate(text[:limit]) if char.isspace()), default=limit)
    if split_at <= 0 or split_at >= len(text):
        split_at = limit
    left = text[:split_at].strip()
    right = text[split_at:].strip()
    if not left or not right:
        split_at = limit
        left, right = text[:split_at].strip(), text[split_at:].strip()
    return split_long_text(left, limit) + split_long_text(right, limit)


def assign_speakers(segments, turns):
    assigned = []
    for segment in segments:
        best_speaker = "Unknown"
        best_overlap = 0.0
        for turn in turns:
            overlap = max(
                0.0,
                min(segment["end"], turn["end"]) - max(segment["start"], turn["start"]),
            )
            if overlap > best_overlap:
                best_overlap = overlap
                best_speaker = turn["speaker"]
        assigned.append({**segment, "speaker": best_speaker})
    return assigned


def group_dialogue(segments, turns):
    assigned = assign_speakers(segments, turns)
    rows = []
    current = None

    def flush():
        nonlocal current
        if current:
            chunks = split_long_text(current["original"])
            duration = max(0.0, current["end"] - current["start"])
            total_chars = max(1, sum(len(chunk) for chunk in chunks))
            offset = 0
            for chunk in chunks:
                start = current["start"] + duration * offset / total_chars
                offset += len(chunk)
                end = current["end"] if offset >= total_chars else current["start"] + duration * offset / total_chars
                rows.append({
                    "id": len(rows),
                    "start": start,
                    "end": end,
                    "speaker": current["speaker"],
                    "original": chunk,
                    "hindi": "",
                })
            current = None

    for segment in assigned:
        text = segment["text"].strip()
        if not text:
            continue
        sentences = split_sentences(text) or [text]
        sentence_total = max(1, sum(len(item) for item in sentences))
        elapsed = 0
        segment_duration = max(0.0, segment["end"] - segment["start"])
        for sentence in sentences:
            start = segment["start"] + segment_duration * elapsed / sentence_total
            elapsed += len(sentence)
            end = segment["end"] if elapsed >= sentence_total else segment["start"] + segment_duration * elapsed / sentence_total
            for part in split_long_text(sentence):
                if len(part) > MAX_LINE_CHARS:
                    raise ValueError("A transcript line could not be split to the 500-character limit.")
                if current and (
                    current["speaker"] != segment["speaker"]
                    or start - current["end"] > 1.2
                    or len(current["original"]) + len(part) + 1 > MAX_LINE_CHARS
                ):
                    flush()
                if current is None:
                    current = {
                        "start": start,
                        "end": end,
                        "speaker": segment["speaker"],
                        "original": part,
                    }
                else:
                    current["end"] = max(current["end"], end)
                    current["original"] = f'{current["original"]} {part}'
                tail = current["original"].rstrip("\"'”’)]}» ")
                if tail and tail[-1] in ".?!…。！？।॥" and not is_numeric_fragment(tail):
                    flush()
    flush()
    return rows


def extract_speaker_references(vocals_path, turns, job_dir):
    audio, sample_rate = sf.read(vocals_path, dtype="float32", always_2d=True)
    mono = audio.mean(axis=1)
    silence = np.zeros(int(0.15 * sample_rate), dtype=np.float32)
    references = {}
    summary = []
    speakers = sorted({turn["speaker"] for turn in turns})
    for speaker in speakers:
        intervals = sorted(
            (turn["start"], turn["end"])
            for turn in turns
            if turn["speaker"] == speaker and turn["end"] - turn["start"] >= MIN_SPEAKER_TURN_SECONDS
        )
        merged = []
        for start, end in intervals:
            if merged and start <= merged[-1][1]:
                merged[-1] = (merged[-1][0], max(merged[-1][1], end))
            else:
                merged.append((start, end))
        long_turns = [item for item in merged if item[1] - item[0] >= REFERENCE_LIMIT_SECONDS]
        if long_turns:
            start, end = max(long_turns, key=lambda item: item[1] - item[0])
            intervals_to_use = [(start, start + REFERENCE_LIMIT_SECONDS)]
        else:
            intervals_to_use = []
            used = 0.0
            for start, end in merged:
                if end - start < MIN_SPEAKER_TURN_SECONDS:
                    continue
                pause = 0.15 if intervals_to_use else 0.0
                remaining = REFERENCE_LIMIT_SECONDS - used - pause
                if remaining <= 0:
                    break
                duration = min(end - start, remaining)
                intervals_to_use.append((start, start + duration))
                used += duration + pause
                if duration < end - start:
                    break
        if not intervals_to_use:
            summary.append([speaker, f"No usable speech (need a {MIN_SPEAKER_TURN_SECONDS:.2f} s turn)", ""])
            continue
        pieces = []
        for index, (start, end) in enumerate(intervals_to_use):
            first = max(0, int(start * sample_rate))
            last = min(len(mono), int(end * sample_rate))
            if last > first:
                if pieces:
                    pieces.append(silence)
                pieces.append(mono[first:last])
        if not pieces:
            summary.append([speaker, "No usable audio", ""])
            continue
        clip = np.concatenate(pieces)[: int(REFERENCE_LIMIT_SECONDS * sample_rate)]
        peak = float(np.max(np.abs(clip))) if len(clip) else 0.0
        if peak > 0:
            clip = clip * (0.9 / peak)
        sample_path = job_dir / f"{speaker.lower()}_reference.wav"
        sf.write(sample_path, clip, sample_rate)
        references[speaker] = str(sample_path)
        duration = len(clip) / sample_rate
        status = f"{duration:.1f} s"
        if duration < REFERENCE_TARGET_SECONDS:
            status += f" (short; {REFERENCE_TARGET_SECONDS:.0f} s recommended)"
        summary.append([speaker, status, str(sample_path)])
    return references, summary


def translate_rows(rows, source_language, context):
    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY is missing. Add it under Kaggle Notebook options > Secrets.")
    if not rows:
        raise ValueError("No dialogue was transcribed.")
    from groq import Groq

    client = Groq(api_key=GROQ_API_KEY)
    translated = {}
    batch_size = 8
    for offset in range(0, len(rows), batch_size):
        batch = rows[offset:offset + batch_size]
        payload = [{"id": row["id"], "text": row["original"]} for row in batch]
        response = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": (
                        f"Translate each {source_language} dialogue line into natural, "
                        "conversational Hindi written in Devanagari. Preserve meaning, "
                        "names, speaker intent, and line IDs. Do not combine lines or "
                        "add explanations. Keep every translation at or below 500 "
                        "characters. Return only a JSON object with a 'translations' "
                        "array; each item must contain 'id' and 'translation'."
                    ),
                },
                {
                    "role": "user",
                    "content": json.dumps(
                        {"context": context or "", "lines": payload},
                        ensure_ascii=False,
                    ),
                },
            ],
            temperature=0.2,
            max_completion_tokens=max(2500, len(batch) * 400),
            response_format={"type": "json_object"},
        )
        content = response.choices[0].message.content or ""
        try:
            result = json.loads(content)
        except json.JSONDecodeError as error:
            raise ValueError(f"Groq returned invalid translation JSON: {error}") from error
        items = result.get("translations") if isinstance(result, dict) else None
        if not isinstance(items, list):
            raise ValueError("Groq response must contain a 'translations' array.")
        expected = {row["id"] for row in batch}
        for item in items:
            if not isinstance(item, dict) or "id" not in item or "translation" not in item:
                raise ValueError("Groq returned a translation without its ID or text.")
            item_id = int(item["id"])
            if item_id not in expected or item_id in translated:
                raise ValueError(f"Groq returned an unexpected or duplicate line ID: {item_id}")
            translation = str(item["translation"]).strip()
            if len(translation) > MAX_LINE_CHARS:
                raise ValueError(f"Groq translation for line {item_id} exceeds 500 characters.")
            translated[item_id] = translation
        missing = expected - translated.keys()
        if missing:
            raise ValueError(f"Groq omitted transcript lines: {sorted(missing)}")
    return [{**row, "hindi": translated[row["id"]]} for row in rows]


def analyze_and_translate(media_path, source_language, whisper_size, context, progress=gr.Progress()):
    if not media_path:
        raise ValueError("Upload a video or audio file first.")
    if not HF_TOKEN:
        raise ValueError("HF_TOKEN is missing. Add it under Kaggle Notebook options > Secrets.")
    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY is missing. Add it under Kaggle Notebook options > Secrets.")

    job_dir = Path(tempfile.mkdtemp(prefix="dubbing_", dir=OUTPUT_ROOT))
    uploaded_path = Path(media_path)
    saved_media = job_dir / f"source{uploaded_path.suffix.lower()}"
    shutil.copy2(uploaded_path, saved_media)
    progress(0.05, desc="Extracting source audio")
    original_audio, is_video = extract_audio(saved_media, job_dir)
    vocals_path, background_path = separate_vocals(original_audio, job_dir, progress)
    raw_segments, detected_language = transcribe(vocals_path, source_language, whisper_size, progress)
    if not raw_segments:
        raise RuntimeError("Whisper returned no speech. Try another source-language setting or a clearer file.")
    turns = diarize(vocals_path, progress)
    rows = group_dialogue(raw_segments, turns)
    if not rows:
        raise RuntimeError("No dialogue rows remained after transcript grouping.")
    reference_map, reference_rows = extract_speaker_references(vocals_path, turns, job_dir)
    progress(0.82, desc="Translating dialogue to Hindi with Groq")
    rows = translate_rows(rows, detected_language, context)
    table = [
        [row["id"], round(row["start"], 2), round(row["end"], 2), row["speaker"], row["original"], row["hindi"], "", "", ""]
        for row in rows
    ]
    state = {
        "job_dir": str(job_dir),
        "media_path": str(saved_media),
        "is_video": is_video,
        "original_audio": str(original_audio),
        "background": str(background_path),
        "vocals": str(vocals_path),
        "rows": rows,
        "references": reference_map,
        "source_language": detected_language,
    }
    progress(1.0, desc="Transcript and translation are ready")
    status = f"{len(rows)} dialogue lines | source language: {detected_language} | edit the Hindi column, then generate the dub."
    return table, reference_rows, status, state

In [ ]:
import uuid

TARGET_LINE_CHARS = 500
HARD_LINE_CHARS = int(TARGET_LINE_CHARS * 1.5)
MAX_LINE_CHARS = HARD_LINE_CHARS


def split_sentences(text):
    endings = set(".?!…。！？।॥")
    closers = set("\"'”’)]}»")
    sentences = []
    start = 0
    index = 0
    while index < len(text):
        if text[index] not in endings:
            index += 1
            continue
        if (
            text[index] == "."
            and index > 0
            and index + 1 < len(text)
            and text[index - 1].isdigit()
            and text[index + 1].isdigit()
        ):
            index += 1
            continue
        end = index + 1
        while end < len(text) and (text[end] in endings or text[end] in closers):
            end += 1
        if end == len(text) or text[end].isspace():
            sentence = text[start:end].strip()
            if sentence:
                sentences.append(sentence)
            start = end
            while start < len(text) and text[start].isspace():
                start += 1
            index = start
        else:
            index += 1
    remainder = text[start:].strip()
    if remainder:
        sentences.append(remainder)
    return sentences


def _split_oversized_sentence(sentence, hard_limit=HARD_LINE_CHARS):
    parts = []
    remaining = sentence.strip()
    marks = set(",，;:；：—–")
    while len(remaining) > hard_limit:
        candidates = []
        for index, char in enumerate(remaining[:hard_limit]):
            if char not in marks:
                continue
            if (
                char in ",，"
                and index > 0
                and index + 1 < len(remaining)
                and remaining[index - 1].isdigit()
                and remaining[index + 1].isdigit()
            ):
                continue
            end = index + 1
            if end < len(remaining) and not remaining[end].isspace():
                continue
            candidates.append(end)
        if candidates:
            split_at = candidates[-1]
        else:
            split_at = max(
                (index for index, char in enumerate(remaining[:hard_limit]) if char.isspace()),
                default=hard_limit,
            )
        if split_at <= 0:
            split_at = hard_limit
        parts.append(remaining[:split_at].strip())
        remaining = remaining[split_at:].strip()
    if remaining:
        parts.append(remaining)
    return parts


def split_long_text(text, limit=TARGET_LINE_CHARS):
    text = text.strip()
    if not text:
        return []
    hard_limit = max(limit, int(limit * 1.5))
    chunks = []
    current = ""
    for sentence in split_sentences(text) or [text]:
        for piece in _split_oversized_sentence(sentence, hard_limit):
            candidate = f"{current} {piece}".strip() if current else piece
            if current and len(candidate) > hard_limit:
                chunks.append(current)
                current = piece
            else:
                current = candidate
            if len(current) >= limit:
                chunks.append(current)
                current = ""
    if current:
        chunks.append(current)
    return chunks


def group_dialogue(segments, turns):
    assigned = assign_speakers(segments, turns)
    rows = []
    current = None

    def flush():
        nonlocal current
        if current:
            rows.append({
                "id": len(rows),
                "start": current["start"],
                "end": current["end"],
                "speaker": current["speaker"],
                "original": current["original"],
                "hindi": "",
            })
            current = None

    for segment in assigned:
        text = segment["text"].strip()
        if not text:
            continue
        sentences = split_sentences(text) or [text]
        total_chars = max(1, sum(len(sentence) for sentence in sentences))
        elapsed = 0
        segment_duration = max(0.0, segment["end"] - segment["start"])
        for sentence in sentences:
            sentence_start = segment["start"] + segment_duration * elapsed / total_chars
            elapsed += len(sentence)
            sentence_end = (
                segment["end"] if elapsed >= total_chars
                else segment["start"] + segment_duration * elapsed / total_chars
            )
            parts = _split_oversized_sentence(sentence)
            part_total = max(1, sum(len(part) for part in parts))
            part_elapsed = 0
            for part in parts:
                start = sentence_start + (sentence_end - sentence_start) * part_elapsed / part_total
                part_elapsed += len(part)
                end = (
                    sentence_end if part_elapsed >= part_total
                    else sentence_start + (sentence_end - sentence_start) * part_elapsed / part_total
                )
                combined_length = len(current["original"]) + 1 + len(part) if current else len(part)
                if current and (
                    current["speaker"] != segment["speaker"]
                    or start - current["end"] > 1.2
                    or combined_length > HARD_LINE_CHARS
                ):
                    flush()
                if current is None:
                    current = {
                        "start": start,
                        "end": end,
                        "speaker": segment["speaker"],
                        "original": part,
                    }
                else:
                    current["end"] = max(current["end"], end)
                    current["original"] = f"{current['original']} {part}"
                if len(current["original"]) >= TARGET_LINE_CHARS:
                    flush()
    flush()
    return rows


HINDI_STYLE_RULES = (
    "You are an expert Hindi-English localization and dubbing writer. Translate the complete meaning into "
    "natural, conversational Hindi/Hinglish that sounds like a real speaker, not a literal or word-for-word translation. "
    "Use Hindi grammar and write Hindi words in Devanagari. When an English word or phrase is more familiar, clearer, "
    "or more natural in everyday speech (especially common, technical, work, and technology terms), keep it in English "
    "using Latin script; do not replace it with an awkward formal Hindi equivalent. Code-switch selectively and naturally: "
    "do not force English into every line, translate every English term, or write Hindi in Latin script. "
    "Prefer simple spoken Hindi over stiff, literary, or heavily Sanskritized wording. Preserve names, brands, acronyms, "
    "and established terms. Preserve every important fact, tense, name, number, negation, question, command, uncertainty, "
    "emotion, and speaker relationship/honorific level. Use context to resolve ambiguity, but never invent, omit, explain, "
    "or add filler or politeness. Keep each input as one complete utterance; do not merge, split, or leave fragments. "
    "Preserve the intended question/exclamation and use natural final punctuation, not a trailing comma. Keep it easy to "
    "say aloud. Accuracy and completeness take priority over a character budget. "
    "Examples: 'I need the file right now.' -> 'मुझे file अभी चाहिए।' (not 'मुझे दस्तावेज़ की तत्काल आवश्यकता है।'); "
    "'Let's check the update.' -> 'चलो update check करते हैं।'; 'We need to fix this issue today.' -> "
    "'हमें आज यह issue fix करना है।'; 'What is your name?' -> 'आपका नाम क्या है?'"
)

ENGLISH_STYLE_RULES = (
    "You are an expert English localization and dubbing writer. Translate the complete meaning into fluent, "
    "natural contemporary spoken English; convey the intent and context, not the source word order or literal phrasing. "
    "Use idiomatic expressions and contractions where appropriate, with natural grammar and phrasing that is easy to say "
    "aloud. Preserve every important fact, tense, name, number, negation, question, command, uncertainty, emotion, and "
    "speaker relationship/formality. Use context to resolve ambiguity, but never invent, omit, explain, or add filler. "
    "Translate Hindi/Hinglish into English rather than copying Hindi words; retain only names or established terms that "
    "are normally left untranslated. Keep each input as one complete utterance; do not merge, split, or leave fragments. "
    "Preserve question/exclamation intent and use appropriate final punctuation, not a trailing comma. Accuracy and "
    "completeness take priority over a character budget."
)


def translate_rows(rows, source_language, context):
    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY is missing. Add it under Kaggle Notebook options > Secrets.")
    if not rows:
        raise ValueError("No dialogue was transcribed.")
    from groq import Groq

    client = Groq(api_key=GROQ_API_KEY)
    translated = {}
    for offset in range(0, len(rows), 8):
        batch = rows[offset:offset + 8]
        response = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": (
                        f"Translate each {source_language} dialogue line into natural Hindi. "
                        f"{HINDI_STYLE_RULES} Aim for at most {MAX_LINE_CHARS} characters when possible, "
                        "but never sacrifice meaning or grammar to meet that budget. Preserve IDs, names, "
                        "speaker intent, and punctuation; "
                        "do not combine lines or add explanations. Return only a JSON object "
                        "with a 'translations' array of {id, translation} items."
                    ),
                },
                {
                    "role": "user",
                    "content": json.dumps(
                        {
                            "context": context or "",
                            "lines": [
                                {"id": row["id"], "text": row["original"]}
                                for row in batch
                            ],
                        },
                        ensure_ascii=False,
                    ),
                },
            ],
            temperature=0.2,
            max_completion_tokens=max(2500, len(batch) * 500),
            response_format={"type": "json_object"},
        )
        content = response.choices[0].message.content or ""
        try:
            result = json.loads(content)
        except json.JSONDecodeError as error:
            raise ValueError(f"Groq returned invalid translation JSON: {error}") from error
        items = result.get("translations") if isinstance(result, dict) else None
        if not isinstance(items, list):
            raise ValueError("Groq response must contain a 'translations' array.")
        expected = {row["id"] for row in batch}
        for item in items:
            if not isinstance(item, dict) or "id" not in item or "translation" not in item:
                raise ValueError("Groq returned a translation without its ID or text.")
            item_id = int(item["id"])
            if item_id not in expected or item_id in translated:
                raise ValueError(f"Groq returned an unexpected or duplicate line ID: {item_id}")
            translation = str(item["translation"]).strip()
            if len(translation) > MAX_LINE_CHARS:
                raise ValueError(
                    f"Groq translation for line {item_id} exceeds {MAX_LINE_CHARS} characters."
                )
            translated[item_id] = translation
        missing = expected - translated.keys()
        if missing:
            raise ValueError(f"Groq omitted transcript lines: {sorted(missing)}")
    return [{**row, "hindi": translated[row["id"]]} for row in rows]


def shorten_hindi_line(text, target_chars, strictness):
    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY is missing. Add it under Kaggle Notebook options > Secrets.")
    from groq import Groq

    response = Groq(api_key=GROQ_API_KEY).chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Rewrite one Hindi dialogue line to be shorter for dubbing while preserving "
                    f"its essential meaning and intent. {HINDI_STYLE_RULES} Keep everyday English "
                    f"phrases in Latin script. Strictness {strictness} of 3: remove more redundancy "
                    f"at each higher level. Return JSON with a 'text' field of at most "
                    f"{target_chars} characters and no explanation."
                ),
            },
            {"role": "user", "content": text},
        ],
        temperature=0.2,
        max_completion_tokens=500,
        response_format={"type": "json_object"},
    )
    content = response.choices[0].message.content or ""
    try:
        result = json.loads(content)
    except json.JSONDecodeError as error:
        raise ValueError(f"Groq returned invalid shortened-text JSON: {error}") from error
    shortened = str(result.get("text", "")).strip() if isinstance(result, dict) else ""
    if not shortened:
        raise ValueError("Groq returned an empty shortened line.")
    return shortened


_analyze_and_translate_base = analyze_and_translate


def analyze_and_translate(media_path, source_language, whisper_size, context, progress=gr.Progress()):
    table, references, status, state = _analyze_and_translate_base(
        media_path, source_language, whisper_size, context, progress
    )
    return table, references, status, state, []

In [ ]:
_TTS_MODEL = None
_TTS_SAMPLE_RATE = None


def load_voxcpm2():
    global _TTS_MODEL, _TTS_SAMPLE_RATE
    if _TTS_MODEL is None:
        from voxcpm import VoxCPM

        _TTS_MODEL = VoxCPM.from_pretrained(
            "openbmb/VoxCPM2",
            load_denoiser=False,
        )
        _TTS_SAMPLE_RATE = int(_TTS_MODEL.tts_model.sample_rate)
    return _TTS_MODEL, _TTS_SAMPLE_RATE


def resample_audio(audio, source_rate, target_rate):
    if source_rate == target_rate:
        return audio.astype(np.float32, copy=False)
    divisor = math.gcd(int(source_rate), int(target_rate))
    return resample_poly(
        audio,
        int(target_rate) // divisor,
        int(source_rate) // divisor,
    ).astype(np.float32)


def trim_silence(audio, sample_rate, db=-40, padding_seconds=0.03):
    peak = float(np.max(np.abs(audio))) if len(audio) else 0.0
    if peak < 1e-6:
        return audio
    active = np.flatnonzero(np.abs(audio) > peak * (10 ** (db / 20)))
    if not len(active):
        return audio
    padding = int(padding_seconds * sample_rate)
    start = max(0, int(active[0]) - padding)
    end = min(len(audio), int(active[-1]) + padding + 1)
    return audio[start:end]


def table_rows(value):
    if hasattr(value, "values"):
        value = value.values.tolist()
    if value is None:
        return []
    return list(value)


def format_srt_time(seconds):
    milliseconds = max(0, round(float(seconds) * 1000))
    hours, remainder = divmod(milliseconds, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    whole_seconds, milliseconds = divmod(remainder, 1000)
    return f"{hours:02}:{minutes:02}:{whole_seconds:02},{milliseconds:03}"


def write_srt(rows, path):
    with open(path, "w", encoding="utf-8") as file:
        number = 0
        for row in rows:
            text = row["hindi"].strip()
            if not text:
                continue
            number += 1
            start = float(row["start"])
            end = max(float(row["end"]), start + 0.25)
            file.write(
                f"{number}\n{format_srt_time(start)} --> {format_srt_time(end)}\n"
                f"{text}\n\n"
            )


def _mono_float_audio(audio):
    if hasattr(audio, "detach"):
        audio = audio.detach().cpu().numpy()
    audio = np.asarray(audio, dtype=np.float32).squeeze()
    if audio.ndim == 2:
        if audio.shape[0] <= 2:
            audio = audio.mean(axis=0)
        elif audio.shape[1] <= 2:
            audio = audio.mean(axis=1)
        else:
            raise RuntimeError("VoxCPM2 returned audio with an unexpected channel shape.")
    if audio.ndim != 1 or not len(audio) or not np.isfinite(audio).all():
        raise RuntimeError("VoxCPM2 returned invalid or empty audio.")
    return trim_silence(audio, _TTS_SAMPLE_RATE)


def _generate_attempt(model, text, reference_path, cfg_value, inference_steps, pace=None, duration_hint=False, slot_seconds=None):
    instructions = []
    if pace:
        instructions.append(pace)
    if duration_hint and slot_seconds:
        instructions.append(f"speak this in {slot_seconds:.1f} seconds")
    prompt = f"({', '.join(instructions)}){text}" if instructions else text
    speech = model.generate(
        text=prompt,
        reference_wav_path=str(reference_path),
        cfg_value=float(cfg_value),
        inference_timesteps=int(inference_steps),
    )
    return _mono_float_audio(speech)


def _write_line_audio(audio, sample_rate, line_dir, row_id):
    versions = []
    for path in line_dir.glob(f"line_{row_id:04d}_v*.wav"):
        try:
            versions.append(int(path.stem.rsplit("_v", 1)[1]))
        except (IndexError, ValueError):
            continue
    version = max(versions, default=0) + 1
    path = line_dir / f"line_{row_id:04d}_v{version}.wav"
    sf.write(path, audio, sample_rate)
    return str(path), version


def _speed_with_ffmpeg(audio, sample_rate, line_dir, row_id, factor):
    temporary_id = uuid.uuid4().hex
    source_path = line_dir / f".line_{row_id:04d}_{temporary_id}.wav"
    output_path = line_dir / f".line_{row_id:04d}_{temporary_id}_fast.wav"
    sf.write(source_path, audio, sample_rate)
    try:
        run_command(
            [
                FFMPEG, "-y", "-v", "error", "-i", str(source_path),
                "-filter:a", f"atempo={factor:.4f}", "-c:a", "pcm_s16le",
                str(output_path),
            ],
            "Speech speed adjustment",
        )
        adjusted, adjusted_rate = sf.read(output_path, dtype="float32")
        if adjusted_rate != sample_rate:
            adjusted = resample_audio(adjusted, adjusted_rate, sample_rate)
        return np.asarray(adjusted, dtype=np.float32)
    finally:
        for path in (source_path, output_path):
            if path.exists():
                path.unlink()


def _fit_generated_line(
    model, text, reference_path, cfg_value, inference_steps, slot_seconds,
    line_dir, row_id, measured_rate, max_speedup, shortening_attempts,
    duration_hint,
):
    speech = _generate_attempt(
        model, text, reference_path, cfg_value, inference_steps,
        duration_hint=duration_hint, slot_seconds=slot_seconds,
    )
    baseline_duration = len(speech) / _TTS_SAMPLE_RATE
    baseline_rate = len(text) / baseline_duration if baseline_duration else 0.0
    current_text = text
    current_duration = baseline_duration
    shortened_count = 0

    if current_duration > slot_seconds:
        speech = _generate_attempt(
            model, current_text, reference_path, cfg_value, inference_steps,
            pace="fast pace", duration_hint=duration_hint, slot_seconds=slot_seconds,
        )
        current_duration = len(speech) / _TTS_SAMPLE_RATE

    rate = measured_rate or baseline_rate
    for strictness in range(1, int(shortening_attempts) + 1):
        if current_duration <= slot_seconds:
            break
        target_chars = max(1, int(rate * slot_seconds * 0.92))
        if target_chars >= len(current_text):
            target_chars = max(1, int(len(current_text) * slot_seconds / current_duration * 0.9))
        candidate = shorten_hindi_line(current_text, target_chars, strictness)
        if len(candidate) >= len(current_text) or len(candidate) > target_chars:
            rate *= 0.8
            continue
        current_text = candidate
        speech = _generate_attempt(
            model, current_text, reference_path, cfg_value, inference_steps,
            pace="fast pace", duration_hint=duration_hint, slot_seconds=slot_seconds,
        )
        current_duration = len(speech) / _TTS_SAMPLE_RATE
        shortened_count += 1

    speed_factor = 1.0
    if current_duration > slot_seconds and max_speedup > 1.0:
        speed_factor = min(float(max_speedup), current_duration / slot_seconds)
        speech = _speed_with_ffmpeg(speech, _TTS_SAMPLE_RATE, line_dir, row_id, speed_factor)
        current_duration = len(speech) / _TTS_SAMPLE_RATE
    fits = current_duration <= slot_seconds
    audio_path, version = _write_line_audio(speech, _TTS_SAMPLE_RATE, line_dir, row_id)
    return {
        "hindi": current_text,
        "audio_path": audio_path,
        "version": version,
        "duration": current_duration,
        "baseline_rate": baseline_rate,
        "speed_factor": speed_factor if speed_factor > 1.001 else 1.0,
        "shortened_count": shortened_count,
        "fits": fits,
    }


def _normalize_shared_reference(shared_reference, job_dir):
    if not shared_reference:
        return None
    output_path = Path(job_dir) / "shared_reference.wav"
    run_command(
        [
            FFMPEG, "-y", "-v", "error", "-i", str(shared_reference),
            "-t", "15", "-ac", "1", "-ar", "24000",
            "-c:a", "pcm_s16le", str(output_path),
        ],
        "Shared reference conversion",
    )
    return str(output_path)


def _place_existing_lines(records, early_start, max_push_seconds, duration_seconds):
    cursor = 0.0
    placed = []
    for source in sorted(records, key=lambda row: (float(row["start"]), int(row["id"]))):
        row = dict(source)
        original_start = float(row["start"])
        scheduled_start = max(0.0, original_start - float(early_start), cursor)
        push = max(0.0, scheduled_start - original_start)
        next_start = row.get("next_start")
        deadline = (
            min(float(duration_seconds), float(next_start))
            if next_start is not None
            else min(float(duration_seconds), float(row["end"]))
        )
        available = max(0.0, deadline - scheduled_start)
        row["scheduled_start"] = scheduled_start
        row["available_seconds"] = available
        row["shift_seconds"] = push
        duration = max(0.0, float(row.get("duration", 0.0)))
        if not row.get("audio_path"):
            row["status"] = "NO AUDIO — generation did not produce a file"
            row["fits"] = False
        else:
            row["fits"] = push <= float(max_push_seconds) and available > 0.02 and duration <= available
            if row["fits"]:
                row["status"] = "FIT"
            elif available <= 0.02:
                row["status"] = "NO SLOT — audio included; following lines queued"
            elif duration > available:
                row["status"] = f"OVER WINDOW — {duration:.2f}s audio / {available:.2f}s slot; included"
            else:
                row["status"] = f"SHIFTED {push:.2f}s — included; exceeds preferred shift"
            cursor = max(cursor, scheduled_start + duration)
        placed.append(row)
    return placed


def _make_output_files(state, records, output_root):
    source_audio, output_rate = sf.read(
        state["background"], dtype="float32", always_2d=True
    )
    if not len(source_audio):
        raise RuntimeError("The separated background audio is empty.")
    dubbed = np.zeros(len(source_audio), dtype=np.float32)
    for row in records:
        if not row.get("fits") or not row.get("audio_path"):
            continue
        speech, speech_rate = sf.read(row["audio_path"], dtype="float32", always_2d=True)
        speech = resample_audio(speech.mean(axis=1), speech_rate, output_rate)
        start_index = max(0, int(float(row["scheduled_start"]) * output_rate))
        end_index = start_index + len(speech)
        window_end = min(
            len(dubbed),
            int(round((float(row["scheduled_start"]) + float(row["available_seconds"])) * output_rate)),
        )
        if end_index > window_end:
            row["status"] = "TOO LONG — resampled audio exceeds its time window"
            row["fits"] = False
            continue
        if end_index - start_index < len(speech):
            row["status"] = "TOO LONG — audio reaches the end of the source"
            row["fits"] = False
            continue
        dubbed[start_index:end_index] += speech

    hop = max(1, int(0.01 * output_rate))
    frame_count = len(dubbed) // hop
    gain = np.ones(len(dubbed), dtype=np.float32)
    if frame_count > 1:
        envelope = np.sqrt(
            np.mean(dubbed[: frame_count * hop].reshape(frame_count, hop) ** 2, axis=1)
        )
        active = (envelope > 0.01).astype(np.float32)
        smooth = np.convolve(active, np.ones(25, dtype=np.float32) / 25, mode="same")
        frame_gain = 1.0 - 0.45 * np.clip(smooth * 1.5, 0, 1)
        gain = np.interp(
            np.arange(len(dubbed)),
            np.arange(frame_count) * hop + hop // 2,
            frame_gain,
        ).astype(np.float32)
    mix = source_audio * gain[:, None] + dubbed[:, None]
    peak = float(np.max(np.abs(mix))) if mix.size else 0.0
    if peak > 0.97:
        mix *= 0.97 / peak

    suffix = uuid.uuid4().hex[:8]
    mix_path = Path(output_root) / f"dubbed_mix_{suffix}.wav"
    sf.write(mix_path, mix, int(output_rate), subtype="PCM_24")
    srt_path = Path(output_root) / f"dubbed_subtitles_{suffix}.srt"
    write_srt(records, srt_path)
    video_path = None
    if state["is_video"]:
        video_path = Path(output_root) / f"dubbed_video_{suffix}.mp4"
        run_command(
            [
                FFMPEG, "-y", "-v", "error", "-i", state["media_path"],
                "-i", str(mix_path), "-map", "0:v:0", "-map", "1:a:0",
                "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
                "-shortest", str(video_path),
            ],
            "Final video export",
        )
    return str(mix_path), str(srt_path), str(video_path) if video_path else None


def _dialogue_table(table, records):
    record_map = {int(row["id"]): row for row in records}
    output = []
    for values in table_rows(table):
        row = list(values[:6])
        while len(row) < 6:
            row.append("")
        try:
            record = record_map.get(int(float(row[0])))
        except (TypeError, ValueError):
            record = None
        if record:
            row[5] = record.get("hindi", row[5])
            row.extend([
                record.get("status", ""),
                round(float(record.get("duration", 0.0)), 2),
                record.get("audio_path", ""),
            ])
        else:
            row.extend(["Not generated", "", ""])
        output.append(row)
    return output


def _generation_result(table, state, records, early_start, max_push_seconds):
    media_info = sf.info(state["background"])
    records = _place_existing_lines(
        records, early_start, max_push_seconds, media_info.duration
    )
    mix_path, srt_path, video_path = _make_output_files(
        state, records, Path(state["job_dir"])
    )
    fit_count = sum(bool(row.get("fits")) for row in records)
    sped_count = sum(float(row.get("speed_factor", 1.0)) > 1.001 for row in records)
    too_long_count = len(records) - fit_count
    shortened_count = sum(int(row.get("shortened_count", 0)) for row in records)
    shifted = [row for row in records if float(row.get("shift_seconds", 0.0)) > 0.01]
    max_shift = max((float(row["shift_seconds"]) for row in shifted), default=0.0)
    samples = list(state.get("speech_rate_samples", []))
    measured_rate = float(np.median(samples)) if samples else None
    updated_state = {**state, "speech_rate_samples": samples, "line_records": records}
    report = (
        f"Fit: {fit_count} | Sped up: {sped_count} | Too long / no slot: {too_long_count} "
        f"| Shortened: {shortened_count} | Shifted: {len(shifted)} "
        f"(max {max_shift:.2f}s)."
    )
    if measured_rate:
        report += f" Measured Hindi pace: {measured_rate:.2f} characters/s."
    status = "Dub complete. " + report
    return (
        _dialogue_table(table, records),
        records,
        updated_state,
        mix_path,
        video_path,
        video_path,
        mix_path,
        srt_path,
        measured_rate,
        status,
    )


def generate_dub(
    table, state, shared_reference, cfg_value, inference_steps, speech_rate,
    early_start, max_push_seconds, max_speedup, shortening_attempts,
    duration_hint, progress=gr.Progress(),
):
    if not state or not state.get("rows"):
        raise ValueError("Analyze and translate a file before generating the dub.")
    values = table_rows(table)
    if not values:
        raise ValueError("The dialogue table is empty.")
    original_by_id = {row["id"]: row for row in state["rows"]}
    edited = []
    seen = set()
    for values_row in values:
        if len(values_row) < 6:
            continue
        try:
            row_id = int(float(values_row[0]))
        except (TypeError, ValueError):
            raise ValueError("The ID column must remain unchanged.")
        if row_id not in original_by_id or row_id in seen:
            raise ValueError(f"Invalid or duplicate dialogue row ID: {row_id}")
        seen.add(row_id)
        hindi = str(values_row[5] or "").strip()
        if not hindi:
            raise ValueError(f"Hindi text for line {row_id} is empty.")
        if len(hindi) > MAX_LINE_CHARS:
            raise ValueError(f"Hindi text for line {row_id} exceeds {MAX_LINE_CHARS} characters.")
        edited.append({**original_by_id[row_id], "hindi": hindi})
    if not edited:
        raise ValueError("No valid dialogue rows were provided.")

    model, sample_rate = load_voxcpm2()
    job_dir = Path(state["job_dir"])
    line_dir = job_dir / "generated_lines"
    line_dir.mkdir(exist_ok=True)
    shared_path = _normalize_shared_reference(shared_reference, job_dir)
    media_duration = sf.info(state["background"]).duration
    ordered = sorted(edited, key=lambda row: (float(row["start"]), int(row["id"])))
    rate_samples = list(state.get("speech_rate_samples", []))
    effective_rate = float(speech_rate) if speech_rate and float(speech_rate) > 0 else (
        float(np.median(rate_samples)) if rate_samples else None
    )
    cursor = 0.0
    records = []
    for index, row in enumerate(ordered):
        next_start = float(ordered[index + 1]["start"]) if index + 1 < len(ordered) else float(row["end"])
        scheduled_start = max(0.0, float(row["start"]) - float(early_start), cursor)
        shift = max(0.0, scheduled_start - float(row["start"]))
        deadline = min(media_duration, next_start)
        available = max(0.0, deadline - scheduled_start)
        base = {
            **row,
            "next_start": next_start if index + 1 < len(ordered) else None,
            "scheduled_start": scheduled_start,
            "available_seconds": available,
            "shift_seconds": shift,
            "reference_path": shared_path or state["references"].get(row["speaker"]),
        }
        if shift > float(max_push_seconds) or available <= 0.02:
            base.update({
                "duration": 0.0,
                "audio_path": "",
                "speed_factor": 1.0,
                "shortened_count": 0,
                "fits": False,
                "status": f"TOO LONG / NO SLOT — needs {shift:.2f}s push or has no time",
            })
            records.append(base)
            continue
        reference_path = base["reference_path"]
        if not reference_path or not Path(reference_path).is_file():
            raise ValueError(
                f"No reference sample for {row['speaker']}. Upload a shared reference audio file or use a longer source turn."
            )
        progress((index + 1) / len(ordered), desc=f"Generating line {index + 1} of {len(ordered)}")
        generated = _fit_generated_line(
            model, row["hindi"], reference_path, cfg_value, inference_steps,
            available, line_dir, row["id"], effective_rate, max_speedup,
            shortening_attempts, duration_hint,
        )
        rate_samples.append(generated["baseline_rate"])
        effective_rate = (
            float(speech_rate) if speech_rate and float(speech_rate) > 0
            else float(np.median(rate_samples))
        )
        base.update(generated)
        base["fits"] = generated["fits"]
        if base["fits"]:
            cursor = scheduled_start + generated["duration"]
            base["status"] = "FIT"
            if generated["speed_factor"] > 1.001:
                base["status"] += f" · sped up {generated['speed_factor']:.2f}x"
            if generated["shortened_count"]:
                base["status"] += f" · shortened {generated['shortened_count']}x"
        else:
            base["status"] = (
                f"TOO LONG — {generated['duration']:.2f}s audio / {available:.2f}s available; "
                "edit Hindi text"
            )
        records.append(base)
    updated_state = {**state, "speech_rate_samples": rate_samples}
    return _generation_result(table, updated_state, records, early_start, max_push_seconds)


def regenerate_line(
    row_id, table, state, line_records, shared_reference, cfg_value,
    inference_steps, speech_rate, early_start, max_push_seconds, max_speedup,
    shortening_attempts, duration_hint,
):
    if not state or not line_records:
        raise ValueError("Generate the dub before regenerating an individual line.")
    row_id = int(row_id)
    table_row = next(
        (
            row for row in table_rows(table)
            if len(row) >= 6 and int(float(row[0])) == row_id
        ),
        None,
    )
    if table_row is None:
        raise ValueError(f"Could not find dialogue line {row_id}.")
    text = str(table_row[5] or "").strip()
    if not text:
        raise ValueError(f"Hindi text for line {row_id} is empty.")
    if len(text) > MAX_LINE_CHARS:
        raise ValueError(f"Hindi text for line {row_id} exceeds {MAX_LINE_CHARS} characters.")
    original = next((row for row in state["rows"] if int(row["id"]) == row_id), None)
    if original is None:
        raise ValueError(f"Original dialogue line {row_id} was not found.")
    model, _ = load_voxcpm2()
    job_dir = Path(state["job_dir"])
    line_dir = job_dir / "generated_lines"
    line_dir.mkdir(exist_ok=True)
    shared_path = _normalize_shared_reference(shared_reference, job_dir)
    records = [dict(row) for row in line_records]
    records = sorted(records, key=lambda row: (float(row["start"]), int(row["id"])))
    cursor = 0.0
    target_record = None
    target_available = 0.0
    for index, record in enumerate(records):
        next_start = float(records[index + 1]["start"]) if index + 1 < len(records) else float(record["end"])
        scheduled_start = max(0.0, float(record["start"]) - float(early_start), cursor)
        deadline = min(sf.info(state["background"]).duration, next_start)
        available = max(0.0, deadline - scheduled_start)
        if int(record["id"]) == row_id:
            target_record = record
            target_available = available
            target_start = scheduled_start
            target_shift = max(0.0, scheduled_start - float(record["start"]))
            break
        duration = float(record.get("duration", 0.0))
        if record.get("audio_path") and duration <= available and scheduled_start - float(record["start"]) <= float(max_push_seconds):
            cursor = scheduled_start + duration
    if target_record is None:
        raise ValueError(f"Generated audio for line {row_id} was not found.")
    if target_shift > float(max_push_seconds) or target_available <= 0.02:
        raise ValueError(f"Line {row_id} has no available time window; edit its timing or nearby line.")
    reference_path = shared_path or target_record.get("reference_path") or state["references"].get(original["speaker"])
    if not reference_path or not Path(reference_path).is_file():
        raise FileNotFoundError(f"Voice reference for line {row_id} is unavailable.")
    old_rate_samples = list(state.get("speech_rate_samples", []))
    effective_rate = float(speech_rate) if speech_rate and float(speech_rate) > 0 else (
        float(np.median(old_rate_samples)) if old_rate_samples else None
    )
    generated = _fit_generated_line(
        model, text, reference_path, cfg_value, inference_steps, target_available,
        line_dir, row_id, effective_rate, max_speedup, shortening_attempts, duration_hint,
    )
    old_rate_samples.append(generated["baseline_rate"])
    replacement = {
        **original,
        "next_start": target_record.get("next_start"),
        "scheduled_start": target_start,
        "available_seconds": target_available,
        "shift_seconds": target_shift,
        "reference_path": reference_path,
        **generated,
        "fits": generated["fits"],
    }
    if replacement["fits"]:
        replacement["status"] = "FIT"
        if generated["speed_factor"] > 1.001:
            replacement["status"] += f" · sped up {generated['speed_factor']:.2f}x"
        if generated["shortened_count"]:
            replacement["status"] += f" · shortened {generated['shortened_count']}x"
    else:
        replacement["status"] = (
            f"TOO LONG — {generated['duration']:.2f}s audio / "
            f"{target_available:.2f}s available; edit Hindi text"
        )
    records = [
        replacement if int(row["id"]) == row_id else row
        for row in records
    ]
    updated_state = {**state, "speech_rate_samples": old_rate_samples}
    return _generation_result(table, updated_state, records, early_start, max_push_seconds)


def run_pace_experiment(sentence, state, reference_audio, cfg_value, inference_steps):
    if not state or not state.get("references"):
        raise ValueError("Analyze media first so the experiment has a speaker reference.")
    sentence = str(sentence or "").strip()
    if not sentence:
        raise ValueError("Enter one Hindi sentence for the pace experiment.")
    reference_path = reference_audio or next(iter(state["references"].values()), None)
    if not reference_path or not Path(reference_path).is_file():
        raise ValueError("No usable voice reference is available for the pace experiment.")
    model, sample_rate = load_voxcpm2()
    conditions = [
        ("No description", ""),
        ("Slow pace", "(slow pace)"),
        ("Fast pace", "(fast pace)"),
        ("Speak in 3 seconds", "(speak this in 3 seconds)"),
    ]
    experiment_dir = Path(tempfile.mkdtemp(prefix="pace_test_", dir=OUTPUT_ROOT))
    results = []
    previews = []
    for label, prefix in conditions:
        durations = []
        first_path = None
        for _ in range(3):
            speech = model.generate(
                text=f"{prefix}{sentence}",
                reference_wav_path=str(reference_path),
                cfg_value=float(cfg_value),
                inference_timesteps=int(inference_steps),
            )
            speech = _mono_float_audio(speech)
            durations.append(len(speech) / sample_rate)
            if first_path is None:
                first_path = experiment_dir / f"{len(results):02d}.wav"
                sf.write(first_path, speech, sample_rate)
        average = float(np.mean(durations))
        results.append((label, average))
        previews.append(str(first_path))
    baseline = results[0][1]
    requested = results[-1][1]
    follows = "near 3 seconds" if abs(requested - 3.0) <= 0.5 else f"{abs(requested - 3.0):.2f}s away from 3s"
    report = "\\n".join(
        f"- {label}: {average:.2f}s average (3 runs)"
        for label, average in results
    )
    report += f"\\n\\nDuration prompt: {follows}; no-description average: {baseline:.2f}s."
    return report, *previews

In [ ]:
import re

LANGUAGE_NAMES = {
    'af': 'Afrikaans',
    'am': 'Amharic',
    'ar': 'Arabic',
    'as': 'Assamese',
    'az': 'Azerbaijani',
    'ba': 'Bashkir',
    'be': 'Belarusian',
    'bg': 'Bulgarian',
    'bn': 'Bengali',
    'bo': 'Tibetan',
    'br': 'Breton',
    'bs': 'Bosnian',
    'ca': 'Catalan',
    'cs': 'Czech',
    'cy': 'Welsh',
    'da': 'Danish',
    'de': 'German',
    'el': 'Greek',
    'en': 'English',
    'es': 'Spanish',
    'et': 'Estonian',
    'eu': 'Basque',
    'fa': 'Persian',
    'fi': 'Finnish',
    'fo': 'Faroese',
    'fr': 'French',
    'gl': 'Galician',
    'gu': 'Gujarati',
    'ha': 'Hausa',
    'haw': 'Hawaiian',
    'he': 'Hebrew',
    'hi': 'Hindi',
    'hr': 'Croatian',
    'ht': 'Haitian Creole',
    'hu': 'Hungarian',
    'hy': 'Armenian',
    'id': 'Indonesian',
    'is': 'Icelandic',
    'it': 'Italian',
    'ja': 'Japanese',
    'jw': 'Javanese',
    'ka': 'Georgian',
    'kk': 'Kazakh',
    'km': 'Khmer',
    'kn': 'Kannada',
    'ko': 'Korean',
    'la': 'Latin',
    'lb': 'Luxembourgish',
    'ln': 'Lingala',
    'lo': 'Lao',
    'lt': 'Lithuanian',
    'lv': 'Latvian',
    'mg': 'Malagasy',
    'mi': 'Maori',
    'mk': 'Macedonian',
    'ml': 'Malayalam',
    'mn': 'Mongolian',
    'mr': 'Marathi',
    'ms': 'Malay',
    'mt': 'Maltese',
    'my': 'Burmese',
    'ne': 'Nepali',
    'nl': 'Dutch',
    'nn': 'Nynorsk',
    'no': 'Norwegian',
    'oc': 'Occitan',
    'pa': 'Punjabi',
    'pl': 'Polish',
    'ps': 'Pashto',
    'pt': 'Portuguese',
    'ro': 'Romanian',
    'ru': 'Russian',
    'sa': 'Sanskrit',
    'sd': 'Sindhi',
    'si': 'Sinhala',
    'sk': 'Slovak',
    'sl': 'Slovenian',
    'sn': 'Shona',
    'so': 'Somali',
    'sq': 'Albanian',
    'sr': 'Serbian',
    'su': 'Sundanese',
    'sv': 'Swedish',
    'sw': 'Swahili',
    'ta': 'Tamil',
    'te': 'Telugu',
    'tg': 'Tajik',
    'th': 'Thai',
    'tk': 'Turkmen',
    'tl': 'Tagalog',
    'tr': 'Turkish',
    'tt': 'Tatar',
    'uk': 'Ukrainian',
    'ur': 'Urdu',
    'uz': 'Uzbek',
    'vi': 'Vietnamese',
    'yi': 'Yiddish',
    'yo': 'Yoruba',
    'zh': 'Chinese',
    'yue': 'Cantonese',
}
VOXCPM2_LANGUAGES = (
    'Arabic',
    'Burmese',
    'Chinese',
    'Danish',
    'Dutch',
    'English',
    'Finnish',
    'French',
    'German',
    'Greek',
    'Hebrew',
    'Hindi',
    'Indonesian',
    'Italian',
    'Japanese',
    'Khmer',
    'Korean',
    'Lao',
    'Malay',
    'Norwegian',
    'Polish',
    'Portuguese',
    'Russian',
    'Spanish',
    'Swahili',
    'Swedish',
    'Tagalog',
    'Thai',
    'Turkish',
    'Vietnamese',
)
PACE_SAMPLES = {
    'Arabic': 'مرحبًا، كيف حالك؟',
    'Burmese': 'မင်္ဂလာပါ၊ နေကောင်းလား။',
    'Chinese': '你好，最近怎么样？',
    'Danish': 'Hej, hvordan går det?',
    'Dutch': 'Hallo, hoe gaat het?',
    'English': 'Hello, how are you?',
    'Finnish': 'Hei, mitä kuuluu?',
    'French': 'Bonjour, comment allez-vous ?',
    'German': 'Hallo, wie geht es dir?',
    'Greek': 'Γεια σου, τι κάνεις;',
    'Hebrew': 'שלום, מה שלומך?',
    'Hindi': 'नमस्ते, आप कैसे हैं?',
    'Indonesian': 'Halo, apa kabar?',
    'Italian': 'Ciao, come stai?',
    'Japanese': 'こんにちは、お元気ですか？',
    'Khmer': 'សួស្តី តើអ្នកសុខសប្បាយទេ?',
    'Korean': '안녕하세요, 잘 지내세요?',
    'Lao': 'ສະບາຍດີ ສະບາຍດີບໍ?',
    'Malay': 'Helo, apa khabar?',
    'Norwegian': 'Hei, hvordan går det?',
    'Polish': 'Cześć, jak się masz?',
    'Portuguese': 'Olá, como está?',
    'Russian': 'Привет, как дела?',
    'Spanish': 'Hola, ¿cómo estás?',
    'Swahili': 'Habari, hujambo?',
    'Swedish': 'Hej, hur mår du?',
    'Tagalog': 'Kamusta, kumusta ka?',
    'Thai': 'สวัสดี สบายดีไหม?',
    'Turkish': 'Merhaba, nasılsın?',
    'Vietnamese': 'Xin chào, bạn khỏe không?',
}
_SENSEVOICE_MODEL = None


class GroqResponseError(ValueError):
    pass


def _is_json_validation_error(error):
    if getattr(error, "code", None) == "json_validate_failed":
        return True
    body = getattr(error, "body", None)
    if isinstance(body, dict):
        detail = body.get("error", body)
        return isinstance(detail, dict) and detail.get("code") == "json_validate_failed"
    return False


def _groq_json(system, payload, token_limit=3000):
    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY is missing. Add it under Kaggle Notebook options > Secrets.")
    from groq import BadRequestError, Groq
    client = Groq(api_key=GROQ_API_KEY)
    last_error = "unknown response shape"
    json_mode = True
    for attempt in range(2):
        current_system = system
        if attempt or not json_mode:
            current_system += "\nReturn exactly one valid JSON object matching the requested schema; do not return an array or prose."
        current_limit = token_limit if attempt == 0 else min(max(token_limit * 2, token_limit + 1000), 6000)
        request = {
            "model": GROQ_MODEL,
            "messages": [{"role": "system", "content": current_system},
                        {"role": "user", "content": json.dumps(payload, ensure_ascii=False)}],
            "temperature": 0.2,
            "max_completion_tokens": current_limit,
        }
        if json_mode:
            request["response_format"] = {"type": "json_object"}
        try:
            response = client.chat.completions.create(**request)
        except BadRequestError as error:
            if not json_mode or not _is_json_validation_error(error):
                raise
            json_mode = False
            last_error = "Groq rejected response_format=json_object (json_validate_failed); retrying with prompt-enforced JSON"
            request.pop("response_format", None)
            request["messages"][0]["content"] += "\nReturn exactly one valid JSON object; do not return an array or prose."
            response = client.chat.completions.create(**request)
        choices = getattr(response, "choices", None)
        if not choices:
            last_error = "response contained no choices"
            continue
        choice = choices[0]
        finish_reason = getattr(choice, "finish_reason", "unknown")
        message = getattr(choice, "message", None)
        content = getattr(message, "content", None)
        if not isinstance(content, str) or not content.strip():
            last_error = f"empty completion (finish_reason={finish_reason})"
            json_mode = False
            continue
        try:
            result = json.loads(content)
        except json.JSONDecodeError as error:
            last_error = f"invalid JSON (finish_reason={finish_reason}): {error}"
            json_mode = False
            continue
        if isinstance(result, dict):
            return result
        last_error = f"expected a JSON object, received {type(result).__name__} (finish_reason={finish_reason})"
        json_mode = False
    raise GroqResponseError(f"Groq returned an invalid JSON response after one retry: {last_error}.")


def _parse_keep_ranges(value, duration):
    ranges = []
    for number, item in enumerate(table_rows(value), 1):
        if not item or all(cell in (None, "") for cell in item[:3]):
            continue
        try:
            start, end = float(item[0]), float(item[1])
            kind = str(item[2] or "other").strip() or "other"
        except (TypeError, ValueError, IndexError) as error:
            raise ValueError(f"Keep-original row {number} needs numeric start/end and a type.") from error
        if not math.isfinite(start) or not math.isfinite(end) or start < 0 or end <= start:
            raise ValueError(f"Keep-original row {number} has an invalid time range.")
        start, end = min(start, duration), min(end, duration)
        if end > start:
            ranges.append({"start": start, "end": end, "type": kind})
    return sorted(ranges, key=lambda item: (item["start"], item["end"]))


def _event_labels(text):
    value = str(text).lower()
    patterns = {
        "laughing": r"laugh|laughter|chuckle|giggl",
        "crying": r"cry|sobb|weeping",
        "sigh": r"sigh|breath",
        "applause": r"applause|clap",
        "singing": r"singing|music vocal",
    }
    return [kind for kind, pattern in patterns.items() if re.search(pattern, value)]


def detect_keep_ranges(vocals_path, threshold=0.5):
    global _SENSEVOICE_MODEL
    try:
        if _SENSEVOICE_MODEL is None:
            from funasr import AutoModel
            _SENSEVOICE_MODEL = AutoModel(
                model="FunAudioLLM/SenseVoiceSmall", hub="hf",
                device="cuda:0" if torch.cuda.is_available() else "cpu",
            )
        audio, rate = sf.read(vocals_path, dtype="float32", always_2d=True)
        mono = resample_audio(audio.mean(axis=1), rate, 16000)
        window, step = 2 * 16000, 10000
        starts = list(range(0, max(1, len(mono) - window + 1), step))
        if not starts or starts[-1] + window < len(mono):
            starts.append(max(0, len(mono) - window))
        clips = [mono[start:min(len(mono), start + window)] for start in starts]
        votes = {}
        for offset in range(0, len(clips), 12):
            batch = clips[offset:offset + 12]
            results = _SENSEVOICE_MODEL.generate(
                input=batch, fs=16000, batch_size=len(batch), language="auto", use_itn=False,
            )
            if not isinstance(results, list) or len(results) != len(batch):
                raise RuntimeError("SenseVoice returned an unexpected batch shape.")
            for index, result in enumerate(results):
                text = result.get("text", "") if isinstance(result, dict) else str(result)
                for kind in _event_labels(text):
                    votes.setdefault(kind, set()).add(offset + index)
        cell, total = 0.25, int(math.ceil(len(mono) / 4000))
        detected = []
        for kind, positive in votes.items():
            active = []
            for slot in range(total):
                mid = (slot + 0.5) * cell
                covering = [i for i, start in enumerate(starts)
                            if start / 16000 <= mid < min(len(mono), start + window) / 16000]
                agreement = sum(i in positive for i in covering) / max(1, len(covering))
                active.append(agreement >= float(threshold))
            begin = None
            for slot, selected in enumerate(active + [False]):
                if selected and begin is None:
                    begin = slot
                elif not selected and begin is not None:
                    detected.append([round(begin * cell, 2), round(slot * cell, 2), kind])
                    begin = None
        return sorted(detected), "SenseVoice threshold is an overlapping-window agreement heuristic, not calibrated confidence. Hindi detection is unvalidated and outside the listed language coverage; review/edit detections."
    except Exception as error:
        return [], f"Auto sound detection is off: {error}. You can still add keep-original ranges manually."


def transcribe(vocals_path, language, model_size, progress):
    from faster_whisper import WhisperModel
    requested = None if language == "Auto-detect" else language
    def run(device, compute_type):
        model = WhisperModel(model_size, device=device, compute_type=compute_type)
        try:
            segments, info = model.transcribe(
                str(vocals_path), language=requested, vad_filter=True, beam_size=5,
                condition_on_previous_text=False, word_timestamps=True,
            )
            rows = []
            for segment in segments:
                text = segment.text.strip()
                if not text:
                    continue
                words = [{"start": float(w.start), "end": float(w.end), "text": w.word}
                         for w in (segment.words or [])]
                rows.append({"start": float(segment.start), "end": float(segment.end),
                             "text": text, "words": words})
            return rows, info.language
        finally:
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    progress(0.35, desc="Transcribing dialogue and word timings")
    if torch.cuda.is_available():
        try:
            return run("cuda", "float16")
        except Exception as error:
            print(f"Whisper GPU inference failed; retrying on CPU. Details: {error}")
    return run("cpu", "int8")


def _ends_sentence(text):
    text = str(text or "").rstrip().rstrip("\"'”’)]}» ")
    return bool(text) and text[-1] in ".?!…。！？।॥؟"


def _add_sentence_punctuation(text, punctuation):
    closers = "\"'”’)]}»"
    end = len(text.rstrip())
    insert_at = end
    while insert_at and text[insert_at - 1] in closers:
        insert_at -= 1
    before, after = text[:insert_at].rstrip(), text[insert_at:]
    if before and before[-1] in ",;:，；：":
        before = before[:-1].rstrip()
    if before and before[-1] in ".?!…。！？।॥؟":
        return text
    return f"{before}{punctuation}{after}"


def _restore_sentence_boundaries(segments, turns, source_language):
    assigned = assign_speakers(segments, turns)
    restored = [{**segment, "words": [dict(word) for word in (segment.get("words") or [])]}
                for segment in assigned]
    pending = []

    def finish_pending():
        nonlocal pending
        if not pending:
            return
        words = [{"id": index, "text": word["text"],
                  "gap_before_seconds": round(max(0.0, word["start"] - pending[index - 1]["end"]), 2) if index else 0.0}
                 for index, word in enumerate(pending)]
        result = _groq_json(
            "Restore sentence boundaries for this unpunctuated speech. Return JSON with sentence_ends, an ordered array of {word_id, punctuation} objects. Use only input word IDs, do not rewrite or add words, do not split at commas, and end each sentence with ., ?, or !. Use pauses as evidence but do not split where the sentence continues. The last word must be included as a sentence end.",
            {"source_language": source_language, "words": words}, 1200)
        ends = result.get("sentence_ends")
        if not isinstance(ends, list) or not ends:
            raise ValueError("Groq sentence-boundary response must contain a non-empty sentence_ends array.")
        previous_id = -1
        for item in ends:
            if not isinstance(item, dict):
                raise ValueError("Groq returned a malformed sentence boundary.")
            try:
                word_id = int(item.get("word_id", -1))
            except (TypeError, ValueError) as error:
                raise ValueError("Groq returned a non-numeric sentence-boundary word ID.") from error
            punctuation = str(item.get("punctuation", ""))
            if word_id <= previous_id or word_id < 0 or word_id >= len(pending):
                raise ValueError("Groq returned sentence-boundary IDs that are unordered or out of range.")
            if punctuation not in {".", "?", "!"}:
                raise ValueError("Groq sentence boundaries must end with '.', '?', or '!'.")
            pending[word_id]["sentence_end"] = True
            pending[word_id]["text"] = _add_sentence_punctuation(pending[word_id]["text"], punctuation)
            previous_id = word_id
        if previous_id != len(pending) - 1:
            raise ValueError("Groq sentence-boundary response did not include the final word.")
        pending = []

    for segment in restored:
        words = segment.get("words") or []
        if not words:
            finish_pending()
            continue
        for word in words:
            start, end = float(word["start"]), float(word["end"])
            speaker, best_overlap = segment["speaker"], 0.0
            for turn in turns:
                overlap = max(0.0, min(end, turn["end"]) - max(start, turn["start"]))
                if overlap > best_overlap:
                    speaker, best_overlap = turn["speaker"], overlap
            if pending and speaker != pending[-1]["speaker"]:
                finish_pending()
            word.update({"start": start, "end": end, "speaker": speaker, "sentence_end": False})
            pending.append(word)
            if _ends_sentence(word["text"]):
                word["sentence_end"] = True
                pending = []
    finish_pending()
    return restored


def group_dialogue(segments, turns, keep_ranges=None):
    assigned, ranges, rows, current = assign_speakers(segments, turns), keep_ranges or [], [], None

    def flush():
        nonlocal current
        if current and current["original"].strip():
            current.update({"id": len(rows), "hindi": "", "translation": ""})
            rows.append(current)
        current = None

    for segment in assigned:
        words = segment.get("words") or []
        if not words:
            overlap = any(segment["start"] < r["end"] and segment["end"] > r["start"] for r in ranges)
            if overlap:
                flush()
                continue
            flush()
            text = str(segment.get("text", "")).strip()
            if text:
                rows.append({"id": len(rows), "start": float(segment["start"]), "end": float(segment["end"]),
                             "speaker": segment["speaker"], "original": text, "hindi": "",
                             "translation": ""})
            continue
        for word in words:
            start, end = float(word["start"]), float(word["end"])
            if any(start < r["end"] and end > r["start"] for r in ranges):
                flush()
                continue
            token = str(word["text"])
            if not token.strip():
                continue
            speaker = word.get("speaker", segment["speaker"])
            if current and current["speaker"] != speaker:
                flush()
            if current is None:
                current = {"start": start, "end": end, "speaker": speaker, "original": token.strip()}
            else:
                current["end"] = end
                current["original"] = (current["original"] + token).strip()
            if word.get("sentence_end") or _ends_sentence(token):
                flush()
    flush()
    return rows


def _scene_analysis(rows, context=""):
    transcript = [{"start": round(r["start"], 2), "end": round(r["end"], 2),
                   "speaker": r["speaker"], "text": r["original"]} for r in rows]
    parts = []
    for offset in range(0, len(transcript), 24):
        parts.append(_groq_json(
            "Analyze this transcript part. Return JSON with summary, speaker_profiles (speaker, role, personality, formality, usual_tone, address_style), and scenes (start, end, mood). Do not invent dialogue.",
            {"context": context, "transcript": transcript[offset:offset + 24]}, 2200))
    result = _groq_json(
        "Combine transcript-part analyses into one concise whole-video summary, consistent speaker profiles and chronological scenes with start/end/mood. Return JSON with summary, speaker_profiles, scenes.",
        {"parts": parts}, 2600)
    return str(result.get("summary", "")), result.get("speaker_profiles", []), result.get("scenes", [])


def analyze_for_review(media_path, source_language, whisper_size, context,
                       scene_enabled=True, event_enabled=True, event_threshold=0.5,
                       progress=gr.Progress()):
    if not media_path:
        raise ValueError("Upload a video or audio file first.")
    if not HF_TOKEN:
        raise ValueError("HF_TOKEN is missing. Add it under Kaggle Notebook options > Secrets.")
    job_dir = Path(tempfile.mkdtemp(prefix="dubbing_", dir=OUTPUT_ROOT))
    uploaded = Path(media_path)
    saved = job_dir / f"source{uploaded.suffix.lower()}"
    shutil.copy2(uploaded, saved)
    progress(0.05, desc="Extracting and separating audio")
    original_audio, is_video = extract_audio(saved, job_dir)
    vocals, background = separate_vocals(original_audio, job_dir, progress)
    raw, detected = transcribe(vocals, source_language, whisper_size, progress)
    if not raw:
        raise RuntimeError("Whisper returned no speech. Try another language or clearer media.")
    turns = diarize(vocals, progress)
    raw = _restore_sentence_boundaries(raw, turns, detected)
    rows = group_dialogue(raw, turns)
    references, reference_rows = extract_speaker_references(vocals, turns, job_dir)
    summary, profiles, scenes = ("", [], [])
    if scene_enabled:
        progress(0.7, desc="Summarizing the video and its scenes")
        summary, profiles, scenes = _scene_analysis(rows, context)
    events, event_status = (detect_keep_ranges(vocals, event_threshold) if event_enabled
                            else ([], "Auto sound detection is off; manual ranges are available."))
    state = {"job_dir": str(job_dir), "media_path": str(saved), "is_video": is_video,
             "original_audio": str(original_audio), "background": str(background), "vocals": str(vocals),
             "raw_segments": raw, "turns": turns, "rows": rows, "references": references,
             "source_language": detected, "scene_summary": summary, "speaker_profiles": profiles,
             "scenes": scenes, "keep_ranges": events, "event_status": event_status, "speech_rates": {}}
    table = [[r["id"], round(r["start"], 2), round(r["end"], 2), r["speaker"], r["original"],
              "", "Not translated", "", ""] for r in rows]
    status = f"{len(rows)} transcript lines | source language: {detected}. Review context and sound ranges, then use Step 1 to correct the transcript. {event_status}"
    scene_rows = [[s.get("start", 0), s.get("end", 0), s.get("mood", "")] for s in scenes]
    return (table, reference_rows, status, state, summary, json.dumps(profiles, ensure_ascii=False, indent=2),
            scene_rows, events, gr.Dropdown(choices=sorted(references), value=next(iter(sorted(references)), None)), None)


def _scene_rows(value):
    result = []
    for row in table_rows(value):
        if len(row) < 3 or row[0] in (None, ""):
            continue
        start, end = float(row[0]), float(row[1])
        if not math.isfinite(start) or not math.isfinite(end) or start < 0 or end <= start:
            raise ValueError("Scene rows need valid numeric start/end times.")
        result.append({"start": start, "end": end, "mood": str(row[2] or "neutral")})
    return result


def _measure_rate(state, reference, speaker, target, cfg, steps):
    path = reference or state.get("references", {}).get(speaker)
    if not path or not Path(path).is_file():
        raise ValueError("Choose a speaker reference or upload a shared reference before measuring pace.")
    model, sample_rate = load_voxcpm2()
    sample = PACE_SAMPLES.get(target)
    if not sample:
        raise ValueError(f"Speaking-speed samples are not configured for {target}.")
    samples = [sample] * 5
    rates = []
    for text in samples:
        audio = _mono_float_audio(model.generate(text=text, reference_wav_path=str(path),
                                 cfg_value=float(cfg), inference_timesteps=int(steps)))
        seconds = len(audio) / sample_rate
        if seconds > 0:
            rates.append(len(text) / seconds)
    if not rates:
        raise RuntimeError("The five pace samples did not produce measurable audio.")
    return float(np.median(rates))


def measure_speed(state, reference, speaker, target, cfg, steps):
    if not state:
        raise ValueError("Analyze media first to create speaker references.")
    rate = _measure_rate(state, reference, speaker, target, cfg, steps)
    rates = dict(state.get("speech_rates", {})); rates[target] = rate
    return rate, {**state, "speech_rates": rates, "pace_target": target}, f"Measured {target} pace from five samples: {rate:.2f} characters/s."


def _line_budget(rows, index, rate):
    start = float(rows[index]["start"])
    deadline = float(rows[index + 1]["start"]) if index + 1 < len(rows) else float(rows[index]["end"])
    return max(1, int(rate * max(0.1, deadline - max(0, start - 0.15)) * 0.88))


def _translate_review_rows(rows, source, target, rate, context, summary, profiles, scenes):
    translations = {}
    source_name = LANGUAGE_NAMES.get(source, source)
    if target == "Hindi":
        style = HINDI_STYLE_RULES
    elif target == "English":
        style = ENGLISH_STYLE_RULES
    else:
        style = f"You are an expert {target} localization writer. Use natural, fluent spoken {target}."

    def parse_translation_items(result, expected_ids):
        if not isinstance(result, dict):
            raise GroqResponseError("Groq translation response must be a JSON object.")
        output = result.get("translations")
        if not isinstance(output, list):
            raise GroqResponseError("Groq response must contain a translations array.")
        items_by_id = {}
        for item in output:
            if not isinstance(item, dict):
                raise GroqResponseError("Groq returned a malformed translation item.")
            raw_id = item.get("id")
            try:
                row_id = int(raw_id)
            except (TypeError, ValueError) as error:
                raise GroqResponseError("Groq returned a non-numeric translation line ID.") from error
            if isinstance(raw_id, bool) or row_id not in expected_ids or row_id in items_by_id:
                raise GroqResponseError(f"Groq returned an invalid or duplicate translation line ID: {row_id}.")
            translation = item.get("translation")
            if not isinstance(translation, str) or not translation.strip():
                raise GroqResponseError(f"Groq returned empty text for line {row_id}.")
            items_by_id[row_id] = item
        return items_by_id

    for offset in range(0, len(rows), 8):
        batch, request = rows[offset:offset + 8], []
        for local, row in enumerate(batch):
            index = offset + local
            mood = next((s["mood"] for s in scenes if s["start"] <= row["start"] <= s["end"]), "neutral")
            request.append({"id": row["id"], "speaker": row["speaker"], "text": row["original"],
                            "max_characters": _line_budget(rows, index, rate), "scene_mood": mood,
                            "previous_lines": [r["original"] for r in rows[max(0, index - 2):index]]})
        system = (
            f"Translate each complete source utterance from {source_name} into {target}. {style} "
            "Use video summary, scene, speaker, and previous-line data only for context; do not copy it into the translation. "
            "Keep every input ID unchanged and return exactly one complete translation per input line in a JSON translations "
            "array of {id, translation} items. Aim for each max_characters budget using concise phrasing, but "
            "never truncate, split, or make a line ungrammatical to fit it; a complete, accurate, natural translation is "
            "more important than a character count. Use context to resolve ambiguity without adding unsupported details. "
            "Before responding, check that each line preserves the source meaning, sounds natural in the requested target "
            "language, uses the normal script for that language, and code-switches only when natural. Preserve appropriate "
            "sentence-ending punctuation. Return JSON only."
        )
        payload = {"context": context, "video_summary": summary, "speaker_profiles": profiles,
                   "scenes": scenes, "lines": request}
        expected = {r["id"] for r in batch}
        batch_error = None
        try:
            result = _groq_json(system, payload, 3500)
            items_by_id = parse_translation_items(result, expected)
        except GroqResponseError as error:
            items_by_id, batch_error = {}, str(error)

        request_by_id = {int(item["id"]): item for item in request}
        missing_ids = expected - items_by_id.keys()
        for row_id in sorted(missing_ids):
            single_payload = {**payload, "lines": [request_by_id[row_id]]}
            try:
                single_result = _groq_json(system, single_payload, 1500)
                single_items = parse_translation_items(single_result, {row_id})
                if row_id not in single_items:
                    raise GroqResponseError(f"Groq omitted transcript line {row_id}.")
                items_by_id[row_id] = single_items[row_id]
            except GroqResponseError as error:
                batch_note = f" Batch response issue: {batch_error}." if batch_error else ""
                raise GroqResponseError(
                    f"Groq could not return a valid translation for line {row_id} after an individual retry."
                    f"{batch_note} Retry error: {error}"
                ) from error

        for local, row in enumerate(batch):
            row_id = int(row["id"])
            item = items_by_id[row_id]
            translation = item["translation"].strip()
            mood = next((s["mood"] for s in scenes if s["start"] <= row["start"] <= s["end"]), "neutral")
            translations[row_id] = translation
    return [{**r, "translation": translations[r["id"]], "hindi": translations[r["id"]]}
            for r in rows]


def _apply_transcript_edits(rows, table):
    edits_by_id = {}
    for values in table_rows(table):
        if len(values) < 5:
            continue
        try:
            row_id = int(float(values[0]))
            start, end = float(values[1]), float(values[2])
        except (TypeError, ValueError):
            continue
        if row_id in edits_by_id:
            raise ValueError(f"Dialogue ID {row_id} appears more than once.")
        edits_by_id[row_id] = (start, end, str(values[3] or "").strip(), str(values[4] or "").strip())
    updated, applied = [], 0
    for row in rows:
        edit = edits_by_id.get(int(row["id"]))
        if edit is None:
            updated.append(row)
            continue
        start, end, speaker_name, original_text = edit
        if abs(start - float(row["start"])) > 0.02 or abs(end - float(row["end"])) > 0.02:
            raise ValueError(f"Do not change timing for dialogue line {row['id']}.")
        if not speaker_name:
            raise ValueError(f"Speaker name for dialogue line {row['id']} cannot be empty.")
        if not original_text:
            raise ValueError(f"Original transcript for line {row['id']} is empty. Restore its text or remove the row.")
        original_speaker = row.get("voice_speaker", row["speaker"])
        updated.append({**row, "voice_speaker": original_speaker,
                        "speaker": speaker_name, "original": original_text})
        applied += (original_text != str(row["original"]).strip()
                    or speaker_name != str(row["speaker"]).strip())
    return updated, applied


def _correct_transcript_rows(rows, source, context, summary, profiles, scenes):
    source_name = LANGUAGE_NAMES.get(source, source)
    corrected = {}
    for offset in range(0, len(rows), 8):
        batch = rows[offset:offset + 8]
        request = []
        for index, row in enumerate(batch, offset):
            mood = next((scene["mood"] for scene in scenes
                         if scene["start"] <= row["start"] <= scene["end"]), "neutral")
            request.append({
                "id": int(row["id"]),
                "speaker": row["speaker"],
                "text": row["original"],
                "scene_mood": mood,
                "previous_lines": [item["original"] for item in rows[max(0, index - 2):index]],
            })
        system = (
            f"Edit these ASR transcript lines in {source_name}. Correct likely speech-recognition "
            "errors, spelling, and punctuation using the surrounding dialogue and context. Do not "
            "translate: keep every line in the original language and its normal script. Preserve "
            "the speaker's intended meaning, names, numbers, negation, uncertainty, and tone. Do "
            "not invent, omit, explain, or add dialogue. Keep every input as one line; do not merge "
            "or split lines. If a line is already correct, leave its wording unchanged. Return "
            "exactly one corrected string for every input ID in a JSON object shaped like "
            '{"corrections":[{"id":0,"text":"..."}]}. Return JSON only.'
        )
        result = _groq_json(system, {
            "context": context,
            "video_summary": summary,
            "speaker_profiles": profiles,
            "scenes": scenes,
            "lines": request,
        }, 3500)
        items = result.get("corrections")
        if not isinstance(items, list):
            raise GroqResponseError("Groq transcript correction response must contain a corrections array.")
        expected = {int(row["id"]) for row in batch}
        batch_corrections = {}
        for item in items:
            if not isinstance(item, dict):
                raise GroqResponseError("Groq returned a malformed transcript correction.")
            raw_id = item.get("id")
            try:
                row_id = int(raw_id)
            except (TypeError, ValueError) as error:
                raise GroqResponseError("Groq returned a non-numeric correction line ID.") from error
            text = item.get("text")
            if (isinstance(raw_id, bool) or row_id not in expected or row_id in batch_corrections
                    or not isinstance(text, str) or not text.strip()):
                raise GroqResponseError(f"Groq returned an invalid or duplicate correction for line {row_id}.")
            batch_corrections[row_id] = text.strip()
        missing = expected - batch_corrections.keys()
        if missing:
            raise GroqResponseError(f"Groq omitted transcript corrections for lines: {sorted(missing)}.")
        corrected.update(batch_corrections)
    return [{**row, "original": corrected[int(row["id"])], "translation": ""}
            for row in rows]


def correct_transcript_for_review(table, state, keep_value, context, summary,
                                  profiles_text, scene_value):
    if not state:
        raise ValueError("Analyze media before correcting the transcript.")
    duration = sf.info(state["background"]).duration
    keep = _parse_keep_ranges(keep_value, duration)
    rows = group_dialogue(state["raw_segments"], state["turns"], keep)
    rows, edited_count = _apply_transcript_edits(rows, table)
    if not rows:
        raise ValueError("No speech remains after excluding keep-original ranges.")
    try:
        profiles = json.loads(profiles_text or "[]")
    except json.JSONDecodeError as error:
        raise ValueError(f"Speaker profiles must be valid JSON: {error}") from error
    scenes = _scene_rows(scene_value)
    corrected = _correct_transcript_rows(
        rows, state["source_language"], context, summary, profiles, scenes)
    signature = [(row["start"], row["end"], row["type"]) for row in keep]
    updated = {
        **state,
        "corrected_rows": corrected,
        "corrected_signature": signature,
        "keep_ranges": keep,
        "scene_summary": summary,
        "speaker_profiles": profiles,
        "scenes": scenes,
    }
    table = [[row["id"], round(row["start"], 2), round(row["end"], 2), row["speaker"],
              row["original"], "", "Not translated", "", ""] for row in corrected]
    source_name = LANGUAGE_NAMES.get(state["source_language"], state["source_language"])
    edit_note = f" Applied {edited_count} edited transcript line(s)." if edited_count else ""
    message = (
        f"Step 1 complete: AI corrected {len(corrected)} {source_name} transcript lines. "
        f"Review them, then continue to Step 2.{edit_note}"
    )
    return table, updated, message, [], None, None, None, None, None, ""


def translate_for_review(table, state, target, context, summary, profiles_text, scene_value,
                         keep_value, speaker, reference, cfg, steps, speech_rate,
                         auto_measure):
    if not state:
        raise ValueError("Analyze media before translating.")
    duration = sf.info(state["background"]).duration
    keep = _parse_keep_ranges(keep_value, duration)
    signature = [(row["start"], row["end"], row["type"]) for row in keep]
    if state.get("corrected_signature") != signature or not state.get("corrected_rows"):
        raise ValueError("Complete Step 1: Correct transcript with AI before translating.")
    rows, edited_count = _apply_transcript_edits(state["corrected_rows"], table)
    if not rows:
        raise ValueError("No speech remains after excluding keep-original ranges.")
    rate = state.get("speech_rates", {}).get(target)
    if speech_rate and float(speech_rate) > 0 and state.get("pace_target") in (None, target):
        rate = float(speech_rate)
    rates = dict(state.get("speech_rates", {}))
    if rate is None and auto_measure:
        rate = _measure_rate(state, reference, speaker, target, cfg, steps); rates[target] = rate
    if rate is None:
        raise ValueError("Measure speaking speed or enable automatic measurement before translation.")
    scenes = _scene_rows(scene_value)
    try:
        profiles = json.loads(profiles_text or "[]")
    except json.JSONDecodeError as error:
        raise ValueError(f"Speaker profiles must be valid JSON: {error}") from error
    translated = _translate_review_rows(rows, state["source_language"], target, rate,
                                         context, summary, profiles, scenes)
    table = [[row["id"], round(row["start"], 2), round(row["end"], 2), row["speaker"],
              row["original"], row["translation"], "Not generated", "", ""] for row in translated]
    signature = [(row["start"], row["end"], row["type"]) for row in keep]
    updated = {
        **state,
        "rows": translated,
        "target_language": target,
        "speech_rates": rates,
        "pace_target": target,
        "keep_ranges": keep,
        "keep_signature": signature,
        "scene_summary": summary,
        "speaker_profiles": profiles,
        "scenes": scenes,
    }
    edit_note = f" Applied {edited_count} edited transcript line(s)." if edited_count else ""
    message = (
        f"Step 2 complete: {len(translated)} lines translated to {target}; "
        f"pace {rate:.2f} characters/s. Review them, then click Generate dub.{edit_note}"
    )
    return table, updated, message, [], None, None, None, None, None, "", rate


def _generate_attempt(model, text, reference_path, cfg_value, inference_steps):
    return _mono_float_audio(model.generate(text=text, reference_wav_path=str(reference_path),
                             cfg_value=float(cfg_value), inference_timesteps=int(inference_steps)))


def _fit_review_line(model, text, reference, cfg, steps, slot, folder,
                     row_id, cap, target):
    audio = _generate_attempt(model, text, reference, cfg, steps)
    baseline = len(text) / max(0.01, len(audio) / _TTS_SAMPLE_RATE)
    duration = len(audio) / _TTS_SAMPLE_RATE
    factor = 1.0
    if duration > slot and slot > 0.02 and cap > 1:
        factor = min(float(cap), duration / slot)
        audio = _speed_with_ffmpeg(audio, _TTS_SAMPLE_RATE, folder, row_id, factor)
        duration = len(audio) / _TTS_SAMPLE_RATE
    path, version = _write_line_audio(audio, _TTS_SAMPLE_RATE, folder, row_id)
    return {"translation": text, "hindi": text,
            "audio_path": path, "version": version, "duration": duration,
            "baseline_rate": baseline, "speed_factor": factor if factor > 1.001 else 1.0,
            "fits": duration <= slot}


def _fade_line(audio, sample_rate, seconds=0.025):
    audio = np.asarray(audio, dtype=np.float32).copy()
    count = min(len(audio) // 2, int(sample_rate * seconds))
    if count:
        ramp = np.linspace(0, 1, count, dtype=np.float32)
        shape = (count,) + (1,) * (audio.ndim - 1)
        audio[:count] *= ramp.reshape(shape)
        audio[-count:] *= ramp[::-1].reshape(shape)
    return audio


def _mix_review(state, records, duck, duck_db, match, room, normalize):
    background, out_rate = sf.read(state["background"], dtype="float32", always_2d=True)
    vocals, vocal_rate = sf.read(state["vocals"], dtype="float32", always_2d=True)
    if vocal_rate != out_rate:
        vocals = np.stack([resample_audio(vocals[:, i], vocal_rate, out_rate)
                           for i in range(vocals.shape[1])], axis=1)
    if vocals.shape[1] == 1 and background.shape[1] > 1:
        vocals = np.repeat(vocals, background.shape[1], axis=1)
    vocals = vocals[:len(background)]
    if len(vocals) < len(background):
        vocals = np.pad(vocals, ((0, len(background)-len(vocals)), (0, 0)))
    voice, kept, loud = np.zeros_like(background), np.zeros_like(background), []
    for item in state.get("keep_ranges", []):
        a, b = max(0, int(item["start"]*out_rate)), min(len(background), int(item["end"]*out_rate))
        if b > a:
            kept[a:b] = _fade_line(vocals[a:b], out_rate)
    for row in records:
        if not row.get("audio_path"):
            continue
        clip, clip_rate = sf.read(row["audio_path"], dtype="float32", always_2d=True)
        clip = _fade_line(resample_audio(clip.mean(axis=1), clip_rate, out_rate), out_rate)
        start = max(0, int(row["scheduled_start"]*out_rate)); end = start+len(clip)
        if end > len(background):
            padding = end-len(background)
            background = np.pad(background, ((0,padding),(0,0)))
            vocals = np.pad(vocals, ((0,padding),(0,0)))
            kept = np.pad(kept, ((0,padding),(0,0)))
            voice = np.pad(voice, ((0,padding),(0,0)))
        if match:
            a, b = max(0, int(row["start"]*out_rate)), min(len(vocals), int(row["end"]*out_rate))
            ref = float(np.sqrt(np.mean(vocals[a:b]**2))) if b > a else 0
            rms = float(np.sqrt(np.mean(clip**2))) if len(clip) else 0
            if ref > 1e-5 and rms > 1e-5:
                clip *= min(2.0, max(0.5, ref/rms))
        if room and len(clip) > int(out_rate*0.04):
            delay = int(out_rate*0.035); clip[delay:] += 0.04*clip[:-delay]
        stereo = np.repeat(clip[:, None], background.shape[1], axis=1)
        voice[start:end] += stereo
        bg = background[start:end]
        if np.sqrt(np.mean(bg**2)) > 0.8*max(1e-5, float(np.sqrt(np.mean(stereo**2)))):
            loud.append(int(row["id"]))
    gain = np.ones(len(background), dtype=np.float32)
    if duck and len(voice):
        hop = max(1, int(out_rate*0.01)); count = len(voice)//hop
        if count:
            active = (np.max(np.abs(voice[:count*hop].reshape(count, hop, -1)), axis=(1,2)) > 0.003).astype(np.float32)
            smooth = np.convolve(active, np.ones(21, dtype=np.float32)/21, mode="same")
            frame_gain = 1-(1-10**(-float(duck_db)/20))*np.clip(smooth,0,1)
            gain = np.interp(np.arange(len(background)), np.arange(count)*hop, frame_gain, right=1).astype(np.float32)
    mix = background*gain[:,None] + voice + kept
    peak = float(np.max(np.abs(mix))) if mix.size else 0
    if peak > 0.97:
        mix *= 0.97/peak
    suffix = uuid.uuid4().hex[:8]; folder = Path(state["job_dir"])
    wav = folder/f"dubbed_mix_{suffix}.wav"; sf.write(wav, mix, out_rate, subtype="PCM_24")
    if normalize:
        normalized = folder/f"dubbed_mix_{suffix}_normalized.wav"
        run_command([FFMPEG,"-y","-v","error","-i",str(wav),"-af","loudnorm=I=-16:TP=-1.5:LRA=11",
                     "-c:a","pcm_s24le",str(normalized)],"Final loudness normalization")
        wav = normalized
    srt = folder/f"dubbed_subtitles_{suffix}.srt"
    with open(srt,"w",encoding="utf-8") as file:
        for number,row in enumerate((r for r in records if r.get("translation")),1):
            subtitle_start = float(row.get("scheduled_start",row["start"]))
            subtitle_end = max(subtitle_start+0.25,subtitle_start+float(row.get("duration",0)))
            file.write(f"{number}\\n{format_srt_time(subtitle_start)} --> {format_srt_time(subtitle_end)}\\n{row['translation']}\\n\\n")
    video = None
    if state["is_video"]:
        video = folder/f"dubbed_video_{suffix}.mp4"
        video_duration = _probe_video_duration(state["media_path"])
        output_duration = max(video_duration, sf.info(wav).duration)
        video_tail = max(0.0, output_duration-video_duration)
        aligned_wav = folder/f"dubbed_mix_{suffix}_video_aligned.wav"
        run_command([FFMPEG,"-y","-v","error","-i",str(wav),"-af",
                     f"apad,atrim=duration={output_duration:.9f},asetpts=PTS-STARTPTS",
                     "-ar",str(out_rate),"-c:a","pcm_s24le",str(aligned_wav)],"Audio/video duration alignment")
        wav = aligned_wav
        run_command([FFMPEG,"-y","-v","error","-i",state["media_path"],"-i",str(wav),
                     "-map","0:v:0","-map","1:a:0",
                     "-vf",f"setpts=PTS-STARTPTS,tpad=stop_mode=clone:stop_duration={video_tail:.9f}",
                     "-af","asetpts=PTS-STARTPTS","-fps_mode","passthrough",
                     "-c:v","libx264","-preset","veryfast","-crf","18","-pix_fmt","yuv420p",
                     "-c:a","aac","-b:a","192k","-t",f"{output_duration:.9f}",
                     "-movflags","+faststart",str(video)],"Final video export")
    return str(wav),str(srt),str(video) if video else None,loud


def _final_review(table,state,records,options):
    placed = _place_existing_lines(records,options["early"],options["push"],sf.info(state["background"]).duration)
    wav,srt,video,loud = _mix_review(state,placed,options["duck"],options["duck_db"],options["match"],options["room"],options["normalize"])
    bad = [r for r in placed if not r.get("fits")]
    queued = [r for r in bad if r.get("audio_path")]
    missing = [r for r in placed if not r.get("audio_path")]
    shifted = [r for r in placed if float(r.get("shift_seconds",0)) > 0.01]
    message = f"Fit window {len(placed)-len(bad)}/{len(placed)} | audio included for {len(placed)-len(missing)}/{len(placed)} lines | sped up {sum(float(r.get('speed_factor',1))>1.001 for r in placed)} | queued/over-window {len(queued)} | shifted {len(shifted)}."
    if shifted: message += "\\n\\nShifted lines: " + ", ".join(str(r["id"]) for r in shifted)
    if queued: message += "\\n\\nQueued/over-window lines (audio included): " + ", ".join(str(r["id"]) for r in queued)
    if missing: message += "\\n\\nLines without generated audio: " + ", ".join(str(r["id"]) for r in missing)
    if loud: message += "\\n\\nLoud background under speech: lines " + ", ".join(map(str,loud))
    message += f"\\n\\nKept original ranges: {len(state.get('keep_ranges',[]))}. {state.get('event_status','')}"
    if state.get("keep_ranges"):
        message += " " + "; ".join(f"{r['start']:.2f}-{r['end']:.2f}s {r['type']}" for r in state["keep_ranges"])
    by_id={int(r["id"]):r for r in placed}; output=[]
    for values in table_rows(table):
        row=list(values[:10]); row += [""]*max(0,10-len(row))
        try: record=by_id.get(int(float(row[0])))
        except (TypeError,ValueError): record=None
        if record:
            row[5]=record.get("translation",row[5])
            row[6]=record.get("status",""); row[7]=round(float(record.get("duration",0)),2); row[8]=record.get("audio_path","")
        output.append(row)
    updated={**state,"line_records":placed}
    rate=state.get("speech_rates",{}).get(state.get("target_language"))
    return output,placed,updated,wav,video,video,wav,srt,rate,message


def generate_review_dub(table,state,shared_reference,cfg,steps,early,push,cap,
                        overflow_action,keep_value,duck,duck_db,match,room,normalize,
                        progress=gr.Progress()):
    if not state or not state.get("rows"):
        raise ValueError("Analyze and translate media before generating.")
    keep=_parse_keep_ranges(keep_value,sf.info(state["background"]).duration)
    signature=[(r["start"],r["end"],r["type"]) for r in keep]
    if signature != state.get("keep_signature",[]):
        raise ValueError("Keep-original ranges changed; click Translate again to split speech around them.")
    originals={int(r["id"]):r for r in state["rows"]}; edits=[]; seen=set()
    for value in table_rows(table):
        if len(value)<6: continue
        row_id=int(float(value[0]))
        if row_id in seen or row_id not in originals: raise ValueError(f"Invalid or duplicate dialogue ID: {row_id}")
        seen.add(row_id); source_text=str(value[4] or "").strip(); text=str(value[5] or "").strip()
        if text and not source_text:
            raise ValueError(f"Original transcript for line {row_id} is empty. Restore its text or clear its translation.")
        if text:
            speaker_name = str(value[3] or "").strip()
            if not speaker_name:
                raise ValueError(f"Speaker name for dialogue line {row_id} cannot be empty.")
            if abs(float(value[1]) - float(originals[row_id]["start"])) > 0.02 or abs(float(value[2]) - float(originals[row_id]["end"])) > 0.02:
                raise ValueError(f"Do not change timing for dialogue line {row_id}.")
            edits.append({**originals[row_id],"voice_speaker":originals[row_id].get("voice_speaker", originals[row_id]["speaker"]),
                          "speaker":speaker_name,"original":source_text,"translation":text,"hindi":text})
    if not edits: raise ValueError("No translated dialogue lines are available.")
    model,_=load_voxcpm2(); folder=Path(state["job_dir"])/"generated_lines"; folder.mkdir(exist_ok=True)
    shared=_normalize_shared_reference(shared_reference,state["job_dir"])
    ordered=sorted(edits,key=lambda r:(float(r["start"]),int(r["id"]))); records=[]; cursor=0.0
    media_duration=sf.info(state["background"]).duration
    for index,row in enumerate(ordered):
        nxt=float(ordered[index+1]["start"]) if index+1<len(ordered) else float(row["end"])
        start=max(0.0,float(row["start"])-float(early),cursor); available=max(0.0,min(media_duration,nxt)-start)
        reference=shared or state.get("references",{}).get(row.get("voice_speaker", row["speaker"]))
        if not reference: raise ValueError(f"No voice reference is available for {row['speaker']}.")
        progress((index+1)/len(ordered),desc=f"Generating {index+1}/{len(ordered)}")
        result=_fit_review_line(model,row["translation"],reference,cfg,steps,available,folder,row["id"],
                                cap,state.get("target_language","Hindi"))
        record={**row,**result,"scheduled_start":start,"available_seconds":available,
                "shift_seconds":max(0,start-float(row["start"])),"next_start":nxt}
        if record.get("audio_path"):
            cursor=max(cursor,start+float(record.get("duration",0)))
        record["status"]="FIT" if record.get("fits") else "OVER WINDOW — generated; will be queued"
        records.append(record)
    options={"early":early,"push":push,"duck":duck,"duck_db":duck_db,"match":match,"room":room,"normalize":normalize}
    edited_by_id={int(row["id"]):row for row in edits}
    state={**state,"rows":[edited_by_id.get(int(row["id"]),row) for row in state["rows"]],
           "keep_ranges":keep,"mix_options":options}
    return _final_review(table,state,records,options)


def run_laugh_experiment(enabled, sentence, state, reference, speaker, cfg, steps):
    if not enabled:
        raise ValueError("Enable the experimental laugh comparison in Advanced first.")
    if not state:
        raise ValueError("Analyze media first to create a speaker reference.")
    text = str(sentence or "").strip()
    if not text:
        raise ValueError("Enter a short sentence for the experiment.")
    path = reference or state.get("references", {}).get(speaker)
    if not path:
        raise ValueError("Choose a speaker reference or upload a shared reference.")
    model, rate = load_voxcpm2()
    folder = Path(state["job_dir"]) / "laugh_experiment"
    folder.mkdir(exist_ok=True)
    paths = []
    for index, prompt in enumerate((text, f"(laughing){text}")):
        audio = _mono_float_audio(model.generate(text=prompt, reference_wav_path=str(path),
                                 cfg_value=float(cfg), inference_timesteps=int(steps)))
        output = folder / f"laugh_{index}.wav"
        sf.write(output, audio, rate)
        paths.append(str(output))
    return "Experimental only; crying is not confirmed.", *paths


## Gradio app

Analyze media, use Step 1 to correct the source transcript with AI, then Step 2 to translate into the selected dubbing language. Review the translated lines, then generate and review the dub. Model cells are not executed while editing this notebook.

In [ ]:
_RECONSTRUCT_MAX_PAUSE_SECONDS = 1.5


def _clean_pipeline_dialogue_rows(dialogue_rows):
    if not isinstance(dialogue_rows, (list, tuple)):
        raise ValueError("Dialogue rows must be a list of dictionaries.")
    cleaned, seen = [], set()
    for position, row in enumerate(dialogue_rows):
        if not isinstance(row, dict):
            raise ValueError(f"Dialogue row {position} is not a dictionary.")
        missing = {"id", "start", "end", "speaker", "text", "hindi", "status"} - row.keys()
        if missing:
            raise ValueError(f"Dialogue row {position} is missing keys: {sorted(missing)}")
        try:
            row_id = int(float(row["id"]))
            start, end = float(row["start"]), float(row["end"])
        except (TypeError, ValueError) as error:
            raise ValueError(f"Dialogue row {position} has an invalid ID or timestamp.") from error
        if row_id in seen:
            raise ValueError(f"Dialogue ID {row_id} appears more than once.")
        if not math.isfinite(start) or not math.isfinite(end) or start < 0 or end < start:
            raise ValueError(f"Dialogue row {row_id} has invalid start/end times.")
        speaker, text = str(row["speaker"] or "").strip(), str(row["text"] or "").strip()
        if not speaker or not text:
            raise ValueError(f"Dialogue row {row_id} needs both a speaker and transcript text.")
        if cleaned and start < cleaned[-1]["start"]:
            raise ValueError("Dialogue rows must be ordered by start time before reconstruction.")
        cleaned.append({"id": row_id, "start": start, "end": end, "speaker": speaker,
                        "text": text, "hindi": str(row["hindi"] or ""),
                        "status": str(row["status"] or "")})
        seen.add(row_id)
    return cleaned


def reconstruct_english_lines(dialogue_rows):
    rows = _clean_pipeline_dialogue_rows(dialogue_rows)
    if not rows:
        return []
    payload = [{"idx": index, "speaker": row["speaker"], "start": row["start"],
                "end": row["end"], "text": row["text"]}
               for index, row in enumerate(rows)]
    system = (
        "You are an expert English ASR transcript editor. You receive the complete ordered transcript as a JSON array. "
        "Before grouping anything, consider the dialogue as a whole and infer its meaning, grammar, speaker turns, and "
        "natural spoken flow. ASR rows are arbitrary chunks: words, clauses, names, numbers, and even full sentences may "
        "be split or joined incorrectly, and punctuation at a chunk boundary may be wrong. For every adjacent boundary, "
        "decide from context whether the first row is a complete independent utterance or a fragment/callout that "
        "continues naturally into the next row. Join fragments when doing so creates one more coherent, grammatical "
        "spoken sentence or utterance, even if the first fragment has a period or looks complete by itself. Use meaning, "
        "syntax, references, connectors, conversational intent, speaker identity, and timestamps together; do not rely "
        "on a fixed list of words or memorized examples. Correct obvious ASR errors, names, grammar, capitalization, and "
        "punctuation without inventing, omitting, summarizing, or adding dialogue. Remove false sentence stops between "
        "joined fragments and punctuate the resulting sentence naturally. Keep genuinely independent sentences/turns "
        "separate; same speaker or a short pause alone is not enough reason to merge. Never merge across a speaker change "
        "or a pause longer than 1.5 seconds. A shorter pause is evidence, not an automatic merge rule. Return short, "
        "natural, speakable sentence-sized items, never a paragraph or a chain of independent sentences. Prefer about "
        "14 words or fewer per item and never exceed 22 words. When combining fragments, keep their combined timed span "
        "at or below 8 seconds when possible; split only at a natural sentence/clause boundary, never mid-thought. "
        "Each input has zero-based idx, speaker, start, end, and text. Every idx must occur exactly once and in order in "
        "original_indices. Do not produce timestamps; Python derives each output time from its first and last indices. "
        "this shape: {\"merged_lines\":[{\"text\":\"...\",\"original_indices\":[0,1]}]}"
        
    )
    result = _groq_json(system, payload, 5000)
    groups = result.get("merged_lines")
    if not isinstance(groups, list) or not groups:
        raise GroqResponseError("Groq reconstruction response must contain a non-empty merged_lines array.")
    def validate_groups(candidate_groups):
        reconstructed, position = [], 0
        expected_indices = list(range(len(rows)))
        for group in candidate_groups:
            if not isinstance(group, dict) or not isinstance(group.get("original_indices"), list):
                raise GroqResponseError("Groq returned a malformed reconstructed sentence.")
            raw_indices = group["original_indices"]
            if any(type(value) is not int for value in raw_indices) or not raw_indices:
                raise GroqResponseError("Groq returned an empty or non-integer original_indices list.")
            indices = list(raw_indices)
            expected = expected_indices[position:position + len(indices)]
            if indices != expected:
                raise GroqResponseError("Groq must preserve every input row once, in chronological order.")
            text = group.get("text")
            if not isinstance(text, str) or not text.strip():
                raise GroqResponseError("Groq returned empty text for a reconstructed sentence.")
            first, last = rows[indices[0]], rows[indices[-1]]
            if any(rows[index]["speaker"] != first["speaker"] for index in indices):
                raise GroqResponseError("Groq attempted to merge dialogue across speaker boundaries.")
            if any(rows[right]["start"] - rows[left]["end"] > _RECONSTRUCT_MAX_PAUSE_SECONDS
                   for left, right in zip(indices, indices[1:])):
                raise GroqResponseError("Groq attempted to merge dialogue across a long pause.")
            reconstructed.append({"id": first["id"], "start": first["start"], "end": last["end"],
                                  "speaker": first["speaker"], "text": text.strip(),
                                  "hindi": "", "status": "Reconstructed"})
            position += len(indices)
        if position != len(rows):
            raise GroqResponseError("Groq omitted one or more transcript rows during reconstruction.")
        return reconstructed
    represented = set()
    for group in groups:
        if isinstance(group, dict) and isinstance(group.get("original_indices"), list):
            represented.update(index for index in group["original_indices"]
                               if type(index) is int and 0 <= index < len(rows))
    missing = sorted(set(range(len(rows))) - represented)
    if not missing:
        return validate_groups(groups)
    retry_system = (system + f" The previous response omitted these required input indices: {missing}. "
                   "Reconstruct the complete transcript again. Include every input index exactly once, "
                   "including the final rows, in chronological order. Do not stop before the last input.")
    retry_result = _groq_json(retry_system, payload, 7000)
    retry_groups = retry_result.get("merged_lines")
    if not isinstance(retry_groups, list) or not retry_groups:
        raise GroqResponseError("Groq reconstruction retry did not return a non-empty merged_lines array.")
    retry_represented = set()
    for group in retry_groups:
        if isinstance(group, dict) and isinstance(group.get("original_indices"), list):
            retry_represented.update(index for index in group["original_indices"]
                                     if type(index) is int and 0 <= index < len(rows))
    retry_missing = sorted(set(range(len(rows))) - retry_represented)
    if retry_missing:
        raise GroqResponseError(f"Groq reconstruction retry still omitted input indices: {retry_missing}.")
    try:
        return validate_groups(retry_groups)
    except GroqResponseError as retry_error:
        raise GroqResponseError(f"Groq reconstruction retry failed: {retry_error}") from retry_error


def translate_to_hindi(dialogue_rows):
    rows = _clean_pipeline_dialogue_rows(dialogue_rows)
    if not rows:
        return []
    translated = {}
    batch_size = 16
    system = (
        "Translate each reconstructed English sentence into accurate, natural, concise spoken Hindi/Hinglish. "
        f"{HINDI_STYLE_RULES} Use Devanagari for Hindi words; keep proper nouns and familiar English terms in Latin "
        "script when natural. Each input is one reconstructed sentence: return exactly one concise translation for "
        "that input and never combine adjacent rows. Preserve the full meaning, names, and facts without adding "
        "explanation or filler. Return exactly one "
        "translation for every input idx in a JSON object shaped like "
        "{\"translations\":[{\"idx\":0,\"hindi\":\"...\"}]}. Return JSON only."
    )
    for offset in range(0, len(rows), batch_size):
        batch = rows[offset:offset + batch_size]
        payload = [{"idx": offset + index, "speaker": row["speaker"], "text": row["text"]}
                   for index, row in enumerate(batch)]
        result = _groq_json(system, payload, 3000)
        items = result.get("translations")
        if not isinstance(items, list):
            retry_system = (system + " Your previous response did not follow the required schema. "
                            "Return exactly one JSON object with a top-level 'translations' array. "
                            "Each array item must have integer 'idx' and string 'hindi' fields. "
                            "Do not use another key name, return prose, or omit any input index.")
            result = _groq_json(retry_system, payload, 4000)
            items = result.get("translations")
            if not isinstance(items, list):
                response_keys = ", ".join(sorted(str(key) for key in result)) or "none"
                raise GroqResponseError(
                    "Groq translation response must contain a top-level 'translations' array; "
                    f"the corrected response still returned these keys: {response_keys}."
                )
        expected = set(range(offset, offset + len(batch)))
        batch_translations = {}
        for item in items:
            if not isinstance(item, dict):
                raise GroqResponseError("Groq returned a malformed Hindi translation item.")
            raw_idx = item.get("idx")
            if type(raw_idx) is not int:
                raise GroqResponseError("Groq returned a non-integer translation idx.")
            idx = raw_idx
            hindi = item.get("hindi")
            if (isinstance(raw_idx, bool) or idx not in expected or idx in batch_translations
                    or not isinstance(hindi, str) or not hindi.strip()):
                raise GroqResponseError(f"Groq returned an invalid or duplicate Hindi translation for idx {idx}.")
            batch_translations[idx] = hindi.strip()
        missing = expected - batch_translations.keys()
        if missing:
            raise GroqResponseError(f"Groq omitted Hindi translations for indices: {sorted(missing)}.")
        translated.update(batch_translations)
    return [{"id": row["id"], "start": row["start"], "end": row["end"],
             "speaker": row["speaker"], "text": row["text"],
             "hindi": translated[index], "status": "Translated"}
            for index, row in enumerate(rows)]


def _dialogue_rows_from_ui(table, state):
    if not state or not state.get("rows"):
        raise ValueError("Analyze media before reconstructing or translating dialogue.")
    state_by_id = {int(row["id"]): row for row in state["rows"]}
    rows, display_speakers, seen = [], {}, set()
    for position, values in enumerate(table_rows(table)):
        if isinstance(values, dict):
            row_id = int(float(values["id"]))
            start, end = float(values["start"]), float(values["end"])
            display_speaker = str(values["speaker"] or "").strip()
            text = str(values.get("text", values.get("original", "")) or "").strip()
            hindi = str(values.get("hindi", values.get("translation", "")) or "")
            row_status = str(values.get("status", "") or "")
        else:
            if len(values) < 7:
                raise ValueError(f"Dialogue table row {position} has fewer than seven columns.")
            row_id = int(float(values[0]))
            start, end = float(values[1]), float(values[2])
            display_speaker, text = str(values[3] or "").strip(), str(values[4] or "").strip()
            hindi, row_status = str(values[5] or ""), str(values[6] or "")
        base = state_by_id.get(row_id)
        if base is None:
            raise ValueError(f"Dialogue row {row_id} is not present in the current analysis state.")
        if row_id in seen:
            raise ValueError(f"Dialogue ID {row_id} appears more than once.")
        speaker_id = str(base.get("voice_speaker") or base.get("speaker", display_speaker)).strip()
        rows.append({"id": row_id, "start": start, "end": end, "speaker": speaker_id,
                     "text": text, "hindi": hindi, "status": row_status})
        display_speakers[row_id] = display_speaker or speaker_id
        seen.add(row_id)
    return rows, display_speakers


def _pipeline_table_values(rows, display_speakers):
    return [[row["id"], round(row["start"], 2), round(row["end"], 2),
             display_speakers.get(row["id"], row["speaker"]), row["text"], row["hindi"],
             row["status"], "", ""] for row in rows]


def _pipeline_state(state, rows, display_speakers, translated=False):
    state_by_id = {int(row["id"]): row for row in state["rows"]}
    updated_rows = []
    for row in rows:
        row_id = int(row["id"])
        base = state_by_id.get(row_id)
        if base is None:
            raise ValueError(f"Reconstructed dialogue ID {row_id} is missing from analysis state.")
        speaker_id = base.get("voice_speaker") or base.get("speaker")
        display_speaker = display_speakers.get(row_id, row["speaker"])
        hindi = row["hindi"] if translated else ""
        updated_rows.append({**base, "id": row_id, "start": row["start"], "end": row["end"],
                             "speaker": display_speaker, "voice_speaker": speaker_id,
                             "original": row["text"], "translation": hindi, "hindi": hindi,
                             "status": row["status"]})
    updated = {**state, "rows": updated_rows, "line_records": []}
    if translated:
        updated["target_language"] = "Hindi"
    return updated


def reconstruct_english_lines_for_ui(table, state):
    if not state:
        raise ValueError("Analyze media before reconstructing English sentences.")
    if str(state.get("source_language", "")).lower() not in ("en", "english"):
        raise ValueError("English reconstruction requires English source audio. Select English during analysis.")
    source_rows, display_speakers = _dialogue_rows_from_ui(table, state)
    reconstructed = reconstruct_english_lines(source_rows)
    for row in reconstructed:
        row["speaker"] = display_speakers.get(row["id"], row["speaker"])
    updated_state = _pipeline_state(state, reconstructed, display_speakers)
    message = f"Reconstructed {len(reconstructed)} English sentences. Review them, then click Translate to Hindi."
    return (_pipeline_table_values(reconstructed, display_speakers), updated_state, message,
            [], None, None, None, None, None, "")


def translate_to_hindi_for_ui(table, state, keep_value):
    if not state:
        raise ValueError("Analyze media before translating dialogue.")
    keep = _parse_keep_ranges(keep_value, sf.info(state["background"]).duration)
    rows, display_speakers = _dialogue_rows_from_ui(table, state)
    if any(row["status"] not in ("Reconstructed", "Translated") for row in rows):
        raise ValueError("Click Reconstruct English Sentences before translating to Hindi.")
    translated = translate_to_hindi(rows)
    for row in translated:
        row["speaker"] = display_speakers.get(row["id"], row["speaker"])
    updated_state = _pipeline_state(state, translated, display_speakers, translated=True)
    updated_state["keep_ranges"] = keep
    updated_state["keep_signature"] = [(item["start"], item["end"], item["type"]) for item in keep]
    message = f"Translated {len(translated)} reconstructed sentences to Hindi. You can now generate the dub."
    return (_pipeline_table_values(translated, display_speakers), updated_state, message,
            [], None, None, None, None, None, "", "Hindi")


In [ ]:
def regenerate_review_line(row_id, table, state, line_records, shared_reference, cfg, steps,
                           early, push, cap, overflow_action,
                           keep_value, duck, duck_db, match, room, normalize):
    if not state or not line_records:
        raise ValueError("Generate the dub before regenerating an individual line.")
    row_id = int(row_id)
    original = next((r for r in state["rows"] if int(r["id"]) == row_id), None)
    if original is None:
        raise ValueError(f"Dialogue line {row_id} was not found.")
    table_row = next((r for r in table_rows(table) if len(r) >= 6 and int(float(r[0])) == row_id), None)
    if table_row is None:
        raise ValueError(f"Dialogue row {row_id} is missing from the table.")
    speaker_name = str(table_row[3] or "").strip()
    if not speaker_name:
        raise ValueError(f"Speaker name for dialogue line {row_id} cannot be empty.")
    if abs(float(table_row[1]) - float(original["start"])) > 0.02 or abs(float(table_row[2]) - float(original["end"])) > 0.02:
        raise ValueError(f"Do not change timing for dialogue line {row_id}.")
    text = str(table_row[5] or "").strip()
    if not text:
        raise ValueError(f"Dialogue line {row_id} is empty.")
    records = [dict(r) for r in line_records]
    target = next((r for r in records if int(r["id"]) == row_id), None)
    if target is None:
        raise ValueError(f"Generated record {row_id} was not found.")
    voice_speaker = original.get("voice_speaker", original["speaker"])
    ref = _normalize_shared_reference(shared_reference, state["job_dir"]) or target.get("reference_path") or state["references"].get(voice_speaker)
    if not ref:
        raise ValueError(f"No voice reference is available for {original['speaker']}.")
    model, _ = load_voxcpm2()
    folder = Path(state["job_dir"]) / "generated_lines"
    result = _fit_review_line(model, text, ref, cfg, steps,
                              float(target.get("available_seconds", 0)), folder, row_id,
                              cap,
                              state.get("target_language", "Hindi"))
    target.update({**result, "speaker": speaker_name, "voice_speaker": voice_speaker})
    target["status"] = "FIT" if result["fits"] else (
        "LEFT OUT — exceeds its safe window" if overflow_action == "Leave this line out"
        else f"TOO LONG — {result['duration']:.2f}s / {target.get('available_seconds', 0):.2f}s")
    options = {"early": early, "push": push, "duck": duck, "duck_db": duck_db,
               "match": match, "room": room, "normalize": normalize}
    return _final_review(table, state, records, options)


def run_laugh_experiment(enabled, sentence, state, reference, speaker, cfg, steps):
    if not enabled:
        raise ValueError("Enable the experimental laugh comparison in Advanced first.")
    if not state:
        raise ValueError("Analyze media first to create a speaker reference.")
    text = str(sentence or "").strip()
    if not text:
        raise ValueError("Enter a short sentence for the experiment.")
    path = reference or state.get("references", {}).get(speaker)
    if not path:
        raise ValueError("Choose a speaker reference or upload a shared reference.")
    model, rate = load_voxcpm2()
    folder = Path(state["job_dir"]) / "laugh_experiment"
    folder.mkdir(exist_ok=True)
    paths = []
    for index, prompt in enumerate((text, f"(laughing){text}")):
        audio = _mono_float_audio(model.generate(text=prompt, reference_wav_path=str(path),
                                 cfg_value=float(cfg), inference_timesteps=int(steps)))
        output = folder / f"laugh_{index}.wav"
        sf.write(output, audio, rate)
        paths.append(str(output))
    return "Experimental only; crying is not confirmed. Compare these two clips.", *paths


with gr.Blocks(theme=gr.themes.Soft(), title="AI Dubbing Studio") as demo:
    gr.Markdown("# AI Dubbing Studio\nUpload media, review its context and transcript, then generate a speaker-aware dub.")
    state = gr.State(None)
    audio_rows_state = gr.State([])
    with gr.Row():
        media = gr.File(label="Video or audio", type="filepath",
                        file_types=[".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v", ".mp3", ".wav", ".m4a", ".flac", ".ogg"])
        source_language = gr.Dropdown(
            choices=[("Auto-detect", "Auto-detect")] + [
                (name, code) for code, name in sorted(LANGUAGE_NAMES.items(), key=lambda item: item[1])],
            value="Auto-detect", label="Spoken language")
        target_language = gr.Dropdown(
            choices=sorted(VOXCPM2_LANGUAGES), value="Hindi",
            label="Dub language (VoxCPM2 supported)")
        whisper_size = gr.Dropdown(choices=["small", "base", "medium"], value="small", label="Whisper model")
    context = gr.Textbox(label="Glossary / story context", placeholder="Names, terms, relationship, or tone", lines=2)
    with gr.Accordion("Advanced", open=False):
        with gr.Row():
            scene_analysis = gr.Checkbox(value=True, label="Analyze summary, speakers, and scenes")
            sound_events = gr.Checkbox(value=True, label="Detect non-speech sounds (experimental for Hindi)")
            auto_measure = gr.Checkbox(value=True, label="Measure speaking speed automatically before translation")
        event_threshold = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label="Event window-agreement threshold (heuristic)")
        with gr.Row():
            overflow_action = gr.Dropdown(choices=["Queue every generated line"],
                                          value="Queue every generated line", label="Overlong lines are queued, never omitted")
        with gr.Row():
            duck = gr.Checkbox(value=True, label="Lower background under dubbed speech")
            duck_db = gr.Slider(0.0, 9.0, value=3.0, step=0.5, label="Background reduction (dB)")
            match_loudness = gr.Checkbox(value=True, label="Match dubbed line loudness to original voice")
        with gr.Row():
            room_effect = gr.Checkbox(value=False, label="Add very light room feel")
            normalize = gr.Checkbox(value=False, label="Normalize final loudness")
            laugh_experiment = gr.Checkbox(value=False, label="Enable experimental laugh comparison")
        with gr.Row():
            early_start = gr.Slider(0.0, 0.5, value=0.15, step=0.05, label="Maximum early start (s)")
            max_push = gr.Slider(0.0, 0.5, value=0.2, step=0.05, label="Maximum shift (s)")
            max_speedup = gr.Slider(1.0, 1.25, value=1.2, step=0.01, label="Maximum final speed-up")
    analyze_button = gr.Button("Analyze media", variant="primary")
    status = gr.Markdown("Upload a file and start the analysis.")
    gr.Markdown("### Video context")
    scene_summary = gr.Textbox(label="Editable video summary", lines=3)
    speaker_profiles = gr.Textbox(label="Editable speaker profiles (JSON)", lines=8)
    scene_table = gr.Dataframe(headers=["Start (s)", "End (s)", "Mood"], datatype=["number", "number", "str"],
                               type="array", interactive=True, label="Editable scenes")
    gr.Markdown("### Keep original non-speech sounds\nSenseVoice detections are approximate; review the ranges. Add, edit, or delete rows. If you change ranges, translate again before generating.")
    keep_events = gr.Dataframe(headers=["Start (s)", "End (s)", "Type"], datatype=["number", "number", "str"],
                               type="array", interactive=True, label="Keep original ranges")
    gr.Markdown("Speaker references are assembled from distinct speech turns (up to 10 seconds). VoxCPM recommends 5–30 seconds; shorter available clips are used as-is and marked short, not looped.")
    speaker_references = gr.Dataframe(headers=["Speaker", "Reference duration / status", "Reference file path"],
                                      datatype=["str", "str", "str"], type="array", interactive=False,
                                      label="Speaker voice references")
    with gr.Row():
        speaker_choice = gr.Dropdown(choices=[], label="Reference speaker")
        shared_reference = gr.Audio(label="Optional shared voice reference", type="filepath")
        cfg_value = gr.Slider(1.0, 3.0, value=2.0, step=0.1, label="VoxCPM2 guidance")
        inference_steps = gr.Slider(4, 20, value=10, step=1, label="VoxCPM2 inference steps")
    with gr.Row():
        speech_rate = gr.Number(value=None, label="Measured pace (characters/second)", precision=2)
        measure_button = gr.Button("Measure speaking speed")
        speed_status = gr.Markdown()
    gr.Markdown("Choose any Whisper-recognized spoken language and any VoxCPM2-supported dub language. Complete Step 1 to correct the source transcript, then Step 2 to translate it. Review the translation before generating audio.")
    with gr.Row():
        correct_button = gr.Button("1. Correct transcript with AI")
        translate_button = gr.Button("2. Translate to selected language", variant="primary")
    dialogue = gr.Dataframe(
        headers=["ID", "Start (s)", "End (s)", "Speaker name (edit)", "Original", "Translation (edit before TTS)",
                 "Dub status", "Audio (s)", "Audio file"],
        datatype=["number", "number", "number", "str", "str", "str", "str", "number", "str"],
        type="array", interactive=True, wrap=True, label="Dialogue")
    with gr.Accordion("Pace experiment (12 generations)", open=False):
        experiment_sentence = gr.Textbox(value="Hello, how are you?", label="Test sentence")
        experiment_button = gr.Button("Run pace experiment")
        experiment_report = gr.Markdown()
        with gr.Row():
            experiment_none = gr.Audio(label="No description")
            experiment_slow = gr.Audio(label="Slow pace")
            experiment_fast = gr.Audio(label="Fast pace")
            experiment_duration = gr.Audio(label="Speak in 3 seconds")
    with gr.Accordion("Experimental laugh comparison", open=False):
        laugh_sentence = gr.Textbox(value="That was funny.", label="Test sentence")
        laugh_button = gr.Button("Compare tagged laugh")
        laugh_report = gr.Markdown()
        with gr.Row():
            laugh_plain = gr.Audio(label="Plain speech")
            laugh_tagged = gr.Audio(label="Tagged laughing")
    with gr.Row():
        max_speedup_display = gr.Markdown("Speed-up is capped by the Advanced slider.")
        generate_button = gr.Button("Generate dub", variant="primary")
    generation_status = gr.Markdown()
    @gr.render(inputs=audio_rows_state)
    def render_line_audio(records):
        if not records:
            return
        gr.Markdown("### Individual dialogue audio")
        for record in records:
            with gr.Row():
                gr.Textbox(value=f"Line {record['id']} · {record.get('speaker','')} · {record.get('status','')}\n{record.get('translation', record.get('hindi',''))}",
                           label="Dialogue", lines=3, interactive=False, scale=3)
                if record.get("audio_path"):
                    gr.Audio(value=record["audio_path"], type="filepath", label="Generated line audio", interactive=False)
                    gr.File(value=record["audio_path"], label="Download line WAV", interactive=False)
                else:
                    gr.Markdown("No audio generated for this line.")
                regenerate_button = gr.Button("Regenerate this row", scale=1)
                def regenerate_callback(table_value, job_state, line_records, reference, cfg, steps,
                                        early, push, cap, action, keep,
                                        duck_on, duck_level, match, room, norm, selected_id=record["id"]):
                    return regenerate_review_line(selected_id, table_value, job_state, line_records,
                        reference, cfg, steps, early, push, cap, action, keep,
                        duck_on, duck_level, match, room, norm)
                regenerate_button.click(
                    fn=regenerate_callback,
                    inputs=[dialogue,state,audio_rows_state,shared_reference,cfg_value,inference_steps,
                            early_start,max_push,max_speedup,overflow_action,
                            keep_events,duck,duck_db,match_loudness,room_effect,normalize],
                    outputs=[dialogue,audio_rows_state,state,full_audio,final_video,video_download,audio_download,
                             srt_download,speech_rate,generation_status])
    full_audio = gr.Audio(label="Full dubbed audio", type="filepath", interactive=False)
    audio_download = gr.File(label="Download dubbed WAV")
    final_video = gr.Video(label="Final dubbed video")
    with gr.Row():
        video_download = gr.File(label="Download dubbed MP4")
        srt_download = gr.File(label="Download subtitles (SRT)")
    analyze_button.click(
        fn=analyze_for_review,
        inputs=[media,source_language,whisper_size,context,scene_analysis,sound_events,event_threshold],
        outputs=[dialogue,speaker_references,status,state,scene_summary,speaker_profiles,scene_table,keep_events,speaker_choice,speech_rate])
    measure_button.click(fn=measure_speed, inputs=[state,shared_reference,speaker_choice,target_language,cfg_value,inference_steps],
                         outputs=[speech_rate,state,speed_status])
    correct_button.click(
        fn=correct_transcript_for_review,
        inputs=[dialogue,state,keep_events,context,scene_summary,speaker_profiles,scene_table],
        outputs=[dialogue,state,status,audio_rows_state,full_audio,final_video,video_download,
                 audio_download,srt_download,generation_status])
    translate_button.click(
        fn=translate_for_review,
        inputs=[dialogue,state,target_language,context,scene_summary,speaker_profiles,scene_table,
                keep_events,speaker_choice,shared_reference,cfg_value,inference_steps,speech_rate,auto_measure],
        outputs=[dialogue,state,status,audio_rows_state,full_audio,final_video,video_download,
                 audio_download,srt_download,generation_status,speech_rate])
    generate_button.click(
        fn=generate_review_dub,
        inputs=[dialogue,state,shared_reference,cfg_value,inference_steps,early_start,max_push,
                max_speedup,overflow_action,keep_events,
                duck,duck_db,match_loudness,room_effect,normalize],
        outputs=[dialogue,audio_rows_state,state,full_audio,final_video,video_download,audio_download,srt_download,
                 speech_rate,generation_status])
    experiment_button.click(fn=run_pace_experiment,
        inputs=[experiment_sentence,state,shared_reference,cfg_value,inference_steps],
        outputs=[experiment_report,experiment_none,experiment_slow,experiment_fast,experiment_duration])
    laugh_button.click(fn=run_laugh_experiment,
        inputs=[laugh_experiment,laugh_sentence,state,shared_reference,speaker_choice,cfg_value,inference_steps],
        outputs=[laugh_report,laugh_plain,laugh_tagged])

demo.queue().launch(share=True, debug=True)